# Scraping code bcci

## Scraping match numbers from 1 to 125

In [42]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Configure Chrome
options = Options()

# Headless
options.add_argument("--headless=new")

# Make browser look more real
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

# Helpful on Linux/servers
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

options.add_argument(
    "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/137.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(options=options)

# Hide webdriver flag
driver.execute_script("""
Object.defineProperty(navigator, 'webdriver', {
    get: () => undefined
})
""")

driver.get("https://www.bcci.tv/domestic/319/syed-mushtaq-ali-trophy-elite")

# Wait for match cards to appear
WebDriverWait(driver, 20).until(
    EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, "a.match-btn.mtch-Cards-btn")
    )
)

# Parse page
soup = BeautifulSoup(driver.page_source, "html.parser")

matches = soup.find_all("a", class_="match-btn mtch-Cards-btn")

print(f"Found {len(matches)} matches")

# Debug if needed
print(driver.title)

driver.quit()

match_numbers = []

base_url = "https://www.bcci.tv/"
for match in matches:
    match_numbers.append(match["href"].split('match/')[-1])

print(match_numbers)


Found 125 matches
Syed Mushtaq Ali Trophy Elite 2025-26 | Fixtures, Results, Videos, Stats & Teams | BCCI.tv
['125', '121', '124', '123', '122', '118', '120', '119', '117', '113', '116', '115', '114', '111', '109', '107', '105', '112', '110', '106', '108', '103', '101', '99', '97', '104', '102', '98', '100', '95', '93', '91', '89', '96', '94', '90', '92', '87', '85', '83', '81', '88', '86', '82', '84', '79', '77', '75', '73', '80', '78', '74', '76', '71', '69', '67', '65', '72', '70', '66', '68', '63', '60', '58', '57', '64', '62', '59', '61', '55', '53', '51', '49', '56', '54', '50', '52', '47', '45', '43', '41', '48', '46', '42', '44', '39', '37', '35', '33', '40', '38', '34', '36', '31', '29', '27', '25', '32', '30', '26', '28', '23', '21', '19', '17', '24', '22', '18', '20', '15', '13', '11', '9', '16', '14', '10', '12', '7', '5', '3', '1', '8', '6', '2', '4']


## Scraping match scorecard links

In [ ]:
import requests
import json
import time
import re

series_data = {}

for match_number in reversed(match_numbers):
    match_data = {}
    
    url = "https://www.bcci.tv/domestic/syed-mushtaq-ali-trophy-elite-2025-26/match/"+match_number

    response = requests.get(url)

    soup = BeautifulSoup(response.text, "html.parser")

    html = str(soup)

    match_id = re.search(r'var\s+matchID\s*=\s*"(\d+)"', html).group(1)
    print(match_id)

    for innings_number in [1,2]:

        url = "https://scores.bcci.tv/feeds/"+str(match_id)+"-Innings"+str(innings_number)+".js?format=json"

        print(url)

        r = requests.get(url)

        text = r.text

        start = text.find('{')

        depth = 0
        end = None

        for i in range(start, len(text)):
            if text[i] == '{':
                depth += 1
            elif text[i] == '}':
                depth -= 1

                if depth == 0:
                    end = i
                    break

        json_text = text[start:end+1]

        data = json.loads(json_text)
        data = data["Innings"+str(innings_number)]

        match_data[innings_number] = data
    
    series_data[match_number] = match_data
    print("Added data for match",match_number)
    time.sleep(1)

15733
https://scores.bcci.tv/feeds/15733-Innings1.js?format=json
https://scores.bcci.tv/feeds/15733-Innings2.js?format=json
Added data for match 1
15734
https://scores.bcci.tv/feeds/15734-Innings1.js?format=json
https://scores.bcci.tv/feeds/15734-Innings2.js?format=json
Added data for match 2
15735
https://scores.bcci.tv/feeds/15735-Innings1.js?format=json
https://scores.bcci.tv/feeds/15735-Innings2.js?format=json
Added data for match 3
15736
https://scores.bcci.tv/feeds/15736-Innings1.js?format=json
https://scores.bcci.tv/feeds/15736-Innings2.js?format=json
Added data for match 4
15737
https://scores.bcci.tv/feeds/15737-Innings1.js?format=json
https://scores.bcci.tv/feeds/15737-Innings2.js?format=json
Added data for match 5
15738
https://scores.bcci.tv/feeds/15738-Innings1.js?format=json
https://scores.bcci.tv/feeds/15738-Innings2.js?format=json
Added data for match 6
15739
https://scores.bcci.tv/feeds/15739-Innings1.js?format=json
https://scores.bcci.tv/feeds/15739-Innings2.js?format

## Creating a player list for all the bcci player names

In [ ]:
player_list = {}

for match_number in reversed(match_numbers):
    print("Match"+str(match_number)+":")
    for innings_num in [1,2]:
        data = series_data[match_number][innings_num]
        print("Innings"+str(innings_num)+":")
        batting_team = data["Extras"][0]["BattingTeamName"]
        bowling_team = data["Extras"][0]["BowlingTeamName"]

        batting_card = data["BattingCard"]
        bowling_card = data["BowlingCard"]
        
        for batsman in batting_card:
            if batting_team not in player_list.keys():
                player_list[batting_team] = {'name':[],'id':[]}
            batter_id = batsman['PlayerID'].strip()
            if batter_id not in player_list[batting_team]['id']:
                batter_name = batsman['PlayerName'].title()
                if '(Wk)' in batter_name:
                    batter_name = batter_name.split('(Wk)')[0]
                if '(C)' in batter_name:
                    batter_name = batter_name.split('(C)')[0]
                batter_name = batter_name.strip()
                player_list[batting_team]['name'].append(batter_name)
                player_list[batting_team]['id'].append(batter_id)
            print(batsman)

        for bowler in bowling_card:
            if bowling_team not in player_list.keys():
                player_list[bowling_team] = {'name':[],'id':[]}
            bowler_id = bowler['PlayerID'].strip()
            if bowler_id not in player_list[bowling_team]['id']:
                bowler_name = bowler['PlayerName'].title()
                if '(Wk)' in bowler_name:
                    bowler_name = bowler_name.split('(Wk)')[0]
                if '(C)' in bowler_name:
                    bowler_name = bowler_name.split('(C)')[0]
                bowler_name = bowler_name.strip()
                player_list[bowling_team]['name'].append(bowler_name)
                player_list[bowling_team]['id'].append(bowler_id)
            print(bowler)
        print()

print(player_list)

Match1:
Innings1:
{'MatchID': '15733', 'InningsNo': '1', 'TeamID': '131', 'PlayerID': 'JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'PlayerName': 'SHIVAM CHAUDHARY ', 'PlayerImage': None, 'PlayingOrder': '1', 'BowlerName': 'sairaj b patil', 'OutDesc': 'c Suryansh Shedge b Sairaj B Patil', 'Runs': '11', 'Balls': '23', 'DotBalls': '17', 'DotBallPercentage': '73.91', 'DotBallFrequency': '1.35', 'Ones': '5', 'Twos': '0', 'Threes': '0', 'Fours': '0', 'Sixes': '1', 'BoundaryPercentage': '54.55', 'BoundaryFrequency': '23.00', 'StrikeRate': '47.83', 'MinOver': '0', 'MinStrikerOver': '0', 'WicketNo': '5', 'AgainstFast': '23', 'AgainstSpin': '0', 'AgainstFastPercent': '100.00', 'AgainstSpinPercent': '0.00'}
{'MatchID': '15733', 'InningsNo': '1', 'TeamID': '131', 'PlayerID': 'KQIiiUcVzC0Oeii1yIJeXiGLJbhJOAJ', 'PlayerName': 'S A AHUJA (WK)', 'PlayerImage': None, 'PlayingOrder': '2', 'BowlerName': 'shardul thakur', 'OutDesc': 'c Surya b Shardul Thakur', 'Runs': '1', 'Balls': '4', 'DotBalls': '3', 'DotBallPerc

In [ ]:
print(len(player_list))
for player in player_list:
    print(player,player_list[player])

542
JCULibKVezGQOiJ01ihIIXyJAeiJOci Shivam Chaudhary
KQIiiUcVzC0Oeii1yIJeXiGLJbhJOAJ S A Ahuja
zJLCQiUGyJJeKi1VXIIJeOcib0AiOih Mohammad Saif
JJOzXACeeI1ib0cyiQiLUJGIJiiKOVh Ravi Singh
JbeIILJUJQGJiA0zeKicyOXihCViO1i Ashutosh Sharma
ciOIiyOLJiVAeiUbIzeQKX10GJCiJhJ Shubham Chaubey
20227a4bd4e78d7d11ec9eb102726df R Sharma
A1yhCI0IObKzJicOeJJeLiiQXUVJiiG A R Pandey
GiQi1LUCieIAJJJeiJbcOyOXzVi0hKI Karn Sharma
UIJcCiehyOiziiXKibJQJGIOL1VeJA0 R K Choudhury
yIIJ0KziiVJJhQAeCeiiUiJcbOXO1GL Atal Bihari Rai
GLcJJQIhieyAeKJ1OJ0bOVCiiziXIiU Shardul Thakur
iiiJ0ICJeXI1hyOzLiVJcKAJGbQUieO Tushar U Deshpande
GJIV0ic1eLiiJAJihUCyiOIOXKeJQzb Shivam Dube
JVeJLKO0hiOQcbAG1XzIICJyiiiiJeU Sairaj B Patil
JczhOiibKLGOIiUQJe0VeiAIJi1CJXy Shams Mulani
2017e64696a9aa9c11e78020028d29d Suryansh Shedge
iIiXceibJAeULzOIK0ihyJVOGJ1CQJi Atharva Vinod Ankolekar
202183277a8f67ae11ec9eb102726df Ayush Mhatre
AXeJIbJi1OLGK0UQihJzyiJeOcViIiC Ajinkya Rahane
LJiCJOA0JUQbeyzehK1iiXiGVIiOJcI Surya
QAGJyiicOXJUiJOJI0LiezVheI1iKC

In [ ]:
data = series_data[1][1]
data

{'BattingCard': [{'MatchID': '15733',
   'InningsNo': '1',
   'TeamID': '131',
   'PlayerID': 'JCULibKVezGQOiJ01ihIIXyJAeiJOci',
   'PlayerName': 'SHIVAM CHAUDHARY ',
   'PlayerImage': None,
   'PlayingOrder': '1',
   'BowlerName': 'sairaj b patil',
   'OutDesc': 'c Suryansh Shedge b Sairaj B Patil',
   'Runs': '11',
   'Balls': '23',
   'DotBalls': '17',
   'DotBallPercentage': '73.91',
   'DotBallFrequency': '1.35',
   'Ones': '5',
   'Twos': '0',
   'Threes': '0',
   'Fours': '0',
   'Sixes': '1',
   'BoundaryPercentage': '54.55',
   'BoundaryFrequency': '23.00',
   'StrikeRate': '47.83',
   'MinOver': '0',
   'MinStrikerOver': '0',
   'WicketNo': '5',
   'AgainstFast': '23',
   'AgainstSpin': '0',
   'AgainstFastPercent': '100.00',
   'AgainstSpinPercent': '0.00'},
  {'MatchID': '15733',
   'InningsNo': '1',
   'TeamID': '131',
   'PlayerID': 'KQIiiUcVzC0Oeii1yIJeXiGLJbhJOAJ',
   'PlayerName': 'S A AHUJA (WK)',
   'PlayerImage': None,
   'PlayingOrder': '2',
   'BowlerName': 'shard

In [ ]:
squads = {
  "team1": {
    "team": {
      "teamid": 170,
      "teamname": "Baroda",
      "teamsname": "BRD",
      "isfullmember": False,
      "isassociated": False,
      "isleagueteam": False,
      "iswomenteam": False,
      "isheader": False,
      "isactive": False,
      "teampriority": "",
      "isvideopresent": False,
      "imageid": 172238,
      "countryname": "",
      "belongsto": "",
      "teamcolor": ""
    },
    "players": [
      {
        "player": [
          {
            "id": "14690",
            "name": "Shashwat Rawat",
            "captain": False,
            "role": "Batter",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "9245",
            "name": "V Solanki",
            "captain": True,
            "role": "WK-Batter",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "14408",
            "name": "Shivalik Sharma",
            "captain": False,
            "role": "Batting Allrounder",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "14451",
            "name": "Safvan Patel",
            "captain": False,
            "role": "Bowler",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "IN",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "18697",
            "name": "Bhanu Pania",
            "captain": False,
            "role": "Batter",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "18698",
            "name": "Chintal Gandhi",
            "captain": False,
            "role": "Bowler",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "IN",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "51364",
            "name": "Amit Passi",
            "captain": False,
            "role": "",
            "keeper": True,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "IN",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "13251",
            "name": "Dhruv Patel",
            "captain": False,
            "role": "Batter",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "IN",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "13250",
            "name": "Ninad Rathva",
            "captain": False,
            "role": "Batting Allrounder",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "32422",
            "name": "Mahesh Pithiya",
            "captain": False,
            "role": "Bowling Allrounder",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "52488",
            "name": "Raj Limbani",
            "captain": False,
            "role": "Bowling Allrounder",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 578435,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          }
        ],
        "category": "playing XI"
      },
      {
        "player": [
          {
            "id": "9509",
            "name": "A Sheth",
            "captain": False,
            "role": "Bowling Allrounder",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 154679,
            "countryimageid": 0,
            "playingxichange": "OUT",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "11311",
            "name": "Krunal Pandya",
            "captain": False,
            "role": "Batting Allrounder",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 171069,
            "countryimageid": 0,
            "playingxichange": "OUT",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "10214",
            "name": "Jitesh Sharma",
            "captain": False,
            "role": "WK-Batter",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 226474,
            "countryimageid": 0,
            "playingxichange": "OUT",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "14274",
            "name": "Rasikh Salam",
            "captain": False,
            "role": "Bowler",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "OUT",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "13130",
            "name": "Mitesh Patel",
            "captain": False,
            "role": "WK-Batter",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "9647",
            "name": "Hardik Pandya",
            "captain": False,
            "role": "Batting Allrounder",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 616519,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "1431795",
            "name": "Lakshit Toksiya",
            "captain": False,
            "role": "",
            "keeper": False,
            "teamname": "BRD",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          }
        ],
        "category": "bench"
      }
    ]
  },
  "team2": {
    "team": {
      "teamid": 250,
      "teamname": "Services",
      "teamsname": "SER",
      "isfullmember": False,
      "isassociated": False,
      "isleagueteam": False,
      "iswomenteam": False,
      "isheader": False,
      "isactive": False,
      "teampriority": "",
      "isvideopresent": False,
      "imageid": 172303,
      "countryname": "",
      "belongsto": "",
      "teamcolor": ""
    },
    "players": [
      {
        "player": [
          {
            "id": "10668",
            "name": "Ravi Chauhan",
            "captain": False,
            "role": "Batter",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "10649",
            "name": "Mohit Ahlawat",
            "captain": True,
            "role": "WK-Batter",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "1447577",
            "name": "Kunwar Pathak",
            "captain": False,
            "role": "Batter",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "IN",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "51319",
            "name": "Vineet Dhankhar",
            "captain": False,
            "role": "",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "10298",
            "name": "Nakul Sharma",
            "captain": False,
            "role": "Batter",
            "keeper": True,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "14159",
            "name": "Abhishek Tiwari",
            "captain": False,
            "role": "Batting Allrounder",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "1466103",
            "name": "Sandeep Nishad",
            "captain": False,
            "role": "Bowler",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "IN",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "1466411",
            "name": "Avinash",
            "captain": False,
            "role": "",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "IN",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "36524",
            "name": "Mohit Rathee",
            "captain": False,
            "role": "Bowling Allrounder",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "10645",
            "name": "Pulkit Narang",
            "captain": False,
            "role": "Bowler",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "11823",
            "name": "Vikas Yadav",
            "captain": False,
            "role": "Batter",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          }
        ],
        "category": "playing XI"
      },
      {
        "player": [
          {
            "id": "1465638",
            "name": "Harsh Vardhan",
            "captain": False,
            "role": "",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "OUT",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "1431835",
            "name": "Arun Kumar",
            "captain": False,
            "role": "WK-Batter",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "OUT",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "1447581",
            "name": "Vishal Gaur",
            "captain": False,
            "role": "Bowler",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "OUT",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "9749",
            "name": "Puneet Datey",
            "captain": False,
            "role": "Bowler",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "10374",
            "name": "Vikas Hathwala",
            "captain": False,
            "role": "Bowling Allrounder",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "13117",
            "name": "Nitin Tanwar",
            "captain": False,
            "role": "Batter",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "10371",
            "name": "Gaurav Kochar",
            "captain": False,
            "role": "Batter",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "13908",
            "name": "Mohit Jangra",
            "captain": False,
            "role": "Bowler",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          },
          {
            "id": "51819",
            "name": "Jayant Goyat",
            "captain": False,
            "role": "Bowler",
            "keeper": False,
            "teamname": "SER",
            "isheader": False,
            "imageId": 0,
            "battingStyle": "",
            "bowlingStyle": "",
            "faceimageid": 182026,
            "countryimageid": 0,
            "playingxichange": "",
            "inmatchchange": "",
            "isoverseas": False
          }
        ],
        "category": "bench"
      }
    ]
  },
  "appindex": {
    "seotitle": "Cricket match squads - BRD vs SER Elite Group C,Syed Mushtaq Ali Trophy Elite 2025 | Cricbuzz.com",
    "weburl": "http://www.cricbuzz.com/cricket-match-squads/128616/brd-vs-ser-elite-group-c-syed-mushtaq-ali-trophy-elite-2025"
  }
}

In [ ]:
for player in squads["team1"]["players"][0]["player"]:
    print(player['name'])
for player in squads["team1"]["players"][1]["player"]:
    print(player["name"])

Shashwat Rawat
V Solanki
Shivalik Sharma
Safvan Patel
Bhanu Pania
Chintal Gandhi
Amit Passi
Dhruv Patel
Ninad Rathva
Mahesh Pithiya
Raj Limbani
A Sheth
Krunal Pandya
Jitesh Sharma
Rasikh Salam
Mitesh Patel
Hardik Pandya
Lakshit Toksiya


## Importing cricbuzz squads

In [2]:
import requests

url = "https://Cricbuzz-Official-Cricket-API.proxy-production.allthingsdev.co/series/10493/squads/"

payload = {}
headers = {
   'x-apihub-key': 'wwbuC7LoWiZP3Lw-RojDhbUD0o-TGm5kyV47sIiFHm4O3C1Yf1',
   'x-apihub-host': 'Cricbuzz-Official-Cricket-API.allthingsdev.co',
   'x-apihub-endpoint': 'c4b3ccd2-0bb1-4d94-98c9-b31f389480be'
}

response = requests.request("GET", url, headers=headers, data=payload)

series_squads = response.json()
squad_ids = {}
for squad in series_squads['squads']:
    if 'squadId' in squad:
        squad_ids[squad["squadType"].split(' Squad')[0]] = squad['squadId']

print(squad_ids)

{'Mumbai': 98734, 'Railways': 98745, 'Kerala': 98756, 'Odisha': 98767, 'Andhra': 98778, 'Assam': 98789, 'Chhattisgarh': 98800, 'Vidarbha': 98811, 'Hyderabad': 98822, 'Madhya Pradesh': 98833, 'Bihar': 98844, 'Chandigarh': 98855, 'Jammu and Kashmir': 98866, 'Maharashtra': 98877, 'Goa': 98888, 'Uttar Pradesh': 98899, 'Himachal Pradesh': 98910, 'Punjab': 98921, 'Gujarat': 98932, 'Services': 98943, 'Baroda': 98954, 'Bengal': 98965, 'Haryana': 98976, 'Puducherry': 98987, 'Saurashtra': 98998, 'Tripura': 99009, 'Rajasthan': 99020, 'Tamil Nadu': 99031, 'Delhi': 99042, 'Jharkhand': 99053, 'Karnataka': 99064, 'Uttarakhand': 99075}


In [4]:
import time
import requests

squads = {}

for team in squad_ids.keys():
   squad_id = squad_ids[team]

   squads[team] = {'name':[],'role':[]}

   url = "https://Cricbuzz-Official-Cricket-API.proxy-production.allthingsdev.co/series/10493/squads/"+str(squad_id)

   payload = {}
   headers = {
      'x-apihub-key': 'wwbuC7LoWiZP3Lw-RojDhbUD0o-TGm5kyV47sIiFHm4O3C1Yf1',
      'x-apihub-host': 'Cricbuzz-Official-Cricket-API.allthingsdev.co',
      'x-apihub-endpoint': 'c4b3ccd2-0bb1-4d94-98c9-b31f389480be'
   }

   response = requests.request("GET", url, headers=headers, data=payload)

   squad = response.json()

   for player in squad['player']:
      if 'id' in player.keys():
         player_name = player['name']
         if 'role' not in player.keys():
            player_role = ""
         else:
            player_role = player['role']
         squads[team]['name'].append(player_name)
         squads[team]['role'].append(player_role)
         print(player_name,player_role)
   time.sleep(1)

Ajinkya Rahane Batsman
Ayush Mhatre Batsman
Suryakumar Yadav Batsman
Siddhesh Lad Batsman
Sarfaraz Khan Batsman
Shivam Dube Batting Allrounder
Musheer Khan Batting Allrounder
Suryansh Shedge Batting Allrounder
Shams Mulani Batting Allrounder
Sairaj Patil Batting Allrounder
Shardul Thakur Bowling Allrounder
Atharva Ankolekar Bowling Allrounder
Tanush Kotian Bowling Allrounder
Hardik Tamore WK-Batsman
Tushar Deshpande Bowler
Shubham Chaubey Batsman
Kunal Yadav Batsman
Mohammad Saif Batsman
Akash Pandey Batsman
Akshat R Pandey Batsman
Rahul Sharma Batsman
Navneet Virk Batsman
Ashutosh Sharma Batting Allrounder
Atal Bihari Rai Batting Allrounder
Shivam Chaudhary Bowling Allrounder
Suraj Ahuja WK-Batsman
Ravi Singh WK-Batsman
Kush Marathe Bowler
Karn Sharma Bowler
Raj Choudhary 
Rohan Kunnummal Batsman
Salman Nizar Batsman
Krishna Prasad Batsman
Akhil Scaria Batsman
Muhammed Sharafuddeen Batting Allrounder
Ahammed Imran Bowling Allrounder
Abdul Basith Bowling Allrounder
Sanju Samson WK-Bats

In [14]:
print(len(squads))
for team in squads:
    print(team,":",squads[team])
print(squads)

32
Mumbai : {'name': ['Ajinkya Rahane', 'Ayush Mhatre', 'Suryakumar Yadav', 'Siddhesh Lad', 'Sarfaraz Khan', 'Shivam Dube', 'Musheer Khan', 'Suryansh Shedge', 'Shams Mulani', 'Sairaj Patil', 'Shardul Thakur', 'Atharva Ankolekar', 'Tanush Kotian', 'Hardik Tamore', 'Tushar Deshpande'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'Bowler']}
Railways : {'name': ['Shubham Chaubey', 'Kunal Yadav', 'Mohammad Saif', 'Akash Pandey', 'Akshat R Pandey', 'Rahul Sharma', 'Navneet Virk', 'Ashutosh Sharma', 'Atal Bihari Rai', 'Shivam Chaudhary', 'Suraj Ahuja', 'Ravi Singh', 'Kush Marathe', 'Karn Sharma', 'Raj Choudhary'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsm

## Final cricbuzz and bcci squads

In [1]:
bcci_squads = {'Railways': {'name': ['Shivam Chaudhary', 'S A Ahuja', 'Mohammad Saif', 'Ravi Singh', 'Ashutosh Sharma', 'Shubham Chaubey', 'R Sharma', 'A R Pandey', 'Karn Sharma', 'R K Choudhury', 'Atal Bihari Rai', 'Navneet Virk', 'Upendra Yadav', 'Kunal Yadav', 'K T Marathe'], 'id': ['JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'KQIiiUcVzC0Oeii1yIJeXiGLJbhJOAJ', 'zJLCQiUGyJJeKi1VXIIJeOcib0AiOih', 'JJOzXACeeI1ib0cyiQiLUJGIJiiKOVh', 'JbeIILJUJQGJiA0zeKicyOXihCViO1i', 'ciOIiyOLJiVAeiUbIzeQKX10GJCiJhJ', '20227a4bd4e78d7d11ec9eb102726df', 'A1yhCI0IObKzJicOeJJeLiiQXUVJiiG', 'GiQi1LUCieIAJJJeiJbcOyOXzVi0hKI', 'UIJcCiehyOiziiXKibJQJGIOL1VeJA0', 'yIIJ0KziiVJJhQAeCeiiUiJcbOXO1GL', 'VJ1QiiGe0ziOKCIhJIJOLyiXeicUAbJ', 'iIXOiiJCK0bQJO1zILcVeyAehJiJiGU', 'yAOJiLzUIbXKieJ1IhiVciOJGQCe0Ji', 'OhKzii1cOiyX0CIJVLJAIeiibUJGQeJ']}, 'Mumbai': {'name': ['Shardul Thakur', 'Tushar U Deshpande', 'Shivam Dube', 'Sairaj B Patil', 'Shams Mulani', 'Suryansh Shedge', 'Atharva Vinod Ankolekar', 'Ayush Mhatre', 'Ajinkya Rahane', 'Surya', 'Hardik Tamore', 'Tanush Kotian', 'S N Khan', 'S D Lad', 'Angkrish Raghuvanshi', 'Yashasvi Bhupendra Jaiswal'], 'id': ['GLcJJQIhieyAeKJ1OJ0bOVCiiziXIiU', 'iiiJ0ICJeXI1hyOzLiVJcKAJGbQUieO', 'GJIV0ic1eLiiJAJihUCyiOIOXKeJQzb', 'JVeJLKO0hiOQcbAG1XzIICJyiiiiJeU', 'JczhOiibKLGOIiUQJe0VeiAIJi1CJXy', '2017e64696a9aa9c11e78020028d29d', 'iIiXceibJAeULzOIK0ihyJVOGJ1CQJi', '202183277a8f67ae11ec9eb102726df', 'AXeJIbJi1OLGK0UQihJzyiJeOcViIiC', 'LJiCJOA0JUQbeyzehK1iiXiGVIiOJcI', 'QAGJyiicOXJUiJOJI0LiezVheI1iKCb', 'iJJI10heOeiIOUJXiAGQyiizcJKCVLb', 'iOKQJXOh0eVJICLiJeGIyJ1biiicAzU', 'hIcJI0LJGOiiAU1zQJXJyibKiCVOiee', '2019db59e82fe9bd11e9a30e02726df', 'OXiLecbiiei1J0UJKIGQJhyOVIACziJ']}, 'Chhattisgarh': {'name': ['Ayush Pandey', 'Amandeep Khare', 'Shashank Chandrakar', 'Shashank Singh', 'P M Yadav', 'Ajay Mandal', 'Gagandeep Singh', 'Shubham Agrawal', 'Dev Aditya Singh', 'Sourabh Majumdar', 'Ravi Kiran', 'Sanjeet Desai  (Rp)', 'A G Rao', 'Sanidhya Hurkat  (C Sub)', 'A A Sarvate', 'Amit Kumar Yadav', 'Mayank Verma', 'Mayank Yadav'], 'id': ['iLJhUiAiJQOIiJzyOJebGCVKiX01eIc', 'VJQeGUKJebJcOiOzJ1LIyiCAhiIXii0', 'ichziKeLOJI1yeUJVibiCAXOIiJQ0JG', 'yUKJVeei1XiJAIGhcCQLJi0JOIiibzO', 'QJ1OOKJhiie0bIGzeJicXiJyUAiICLV', '1QyXiKVOibICGLJAUeiihcJIJOJize0', 'OL0QyeJJK1VbiOIJIJceXzCiihAiGUi', '0AiiiOUb1ICieIQJJVJJzKhXyiLGOce', '20181ff1c6e9d04511e8a30e02726df', '202059a13493444b11eb993e02726df', 'hJeKz0CIIJebXiOGJiAOVicQJL1yUii', 'iLCUibiIzJ1h0eiyicKIQOVeXOJJAGJ', 'eOUJeJJCXyGIiz0ii1LiAhVIKQibOJc', 'LJiIiiyiObGecIh0iVXQUzJOe1CJAJK', 'JieUJGbiiILAe0hzciCI1JiOOXJVyKQ', '20252ecea9c9a7ec11f083bd061540e', '201922dfeb3be67211e9a30e02726df', '2017f79a382fd34211e792a5028d29d']}, 'Vidarbha': {'name': ['Umesh Yadav', 'D G Nalkande', 'Yash Kadam', 'Y R Thakur', 'P R Rekhade', 'Harsh Dubey', 'Atharva Taide', 'A Mokhade', 'Y V Rathod', 'Shivam D', 'Varun Raj Singh Bisht', 'Dhruv Shorey', 'Bhute', 'A G Daga', 'A V Wadkar', 'Praful Hinge'], 'id': ['0IOJiGKJUJhVX1CObiLIyeeiQciAizJ', 'bJI1iOCiiKeXIiyOGUiJVJAzcehL0QJ', 'iJXJiOJQIbeiK0UiCLyizchGOAVeIJ1', 'zJehyecIIU0iJiQGXJ1AOiiOLVbCKiJ', 'iiIUiy0JiLbieJOQCcIVJKhGXeO1zAJ', '2016733d4887acd711e6842a028d29d', 'KIGieiAOXJbL1IizVcJiyUJhJ0QiOeC', '20169de53041acd211e6842a028d29d', '0IhUXiOzLye1KiCVJbGQeJAJiicOJIi', '201756a87987d11c11e792a5028d29d', '20198b5bc8dce82a11e9a30e02726df', 'iOiAJeiJQU1bcJIChXyO0GeiiIJKLzV', '20184d9e8c92c22a11e8a30e02726df', 'IJhiJ10OOGiUCcXKViezQiJAeLJibyI', 'CLJizbIiIOAUcGhJQ0JOyeeKiiJVX1i', '201746f067a9d1a711e792a5028d29d']}, 'Maharashtra': {'name': ['Prithvi Shaw', 'Arshin Kulkarni', 'R D Gaikwad', 'R A Tripathi', 'A N Kazi', 'R R Nikam', 'N S Naik', 'Vicky Ostwal', 'R S Ghosh', 'R S Hangargekar', 'P H Solanki', 'Jalaj Saxena', 'M S Bhandari', 'Y S Dongre', 'Niraj Joshi', 'T H Sanghvi', 'Sahil Parakh', 'Y J Nahar'], 'id': ['JOJ0iQOiUJyGiJiAc1IiCLXVehebKzI', '20183cd3a8dcd45411e8a30e02726df', 'ObLiOCyc1eiGVUJhKIzAJJeXJiiQ0iI', 'iceiVGLOJzeAiU0OJ1XhJiybIIJiQKC', 'eUVGOAJQciXICiJiebKOJJLzi0Iy1ih', 'XJKiLIiVcOQJiJeCIbGJ1izOeUyihA0', 'zcJGUKiebXII01QeVJiOAiyiJhCJLOi', '201673fd55f3b24311e6842a028d29d', 'iAJzcLGU01ihVCeieOiQiIJJKJIyXbO', '2016bb914346b22b11e6842a028d29d', 'KihOJCQXbiALIiyJceUOi1VJei0GJIz', 'JVQeJ0eKCUhIOizycbAiiIJ1GLJOXii', '2016e8ff22ab8d8511e69558028d29d', 'OeJiAeKihJyJLIQbVJGicX0OiiUIzC1', '202120b0d9486a0211ec9eb102726df', '2016bc5d0bc7b30e11e6842a028d29d', '2021954c2a436a0911ec9eb102726df', '2019728c9b75303d11e9a30e02726df']}, 'J & K': {'name': ['Auqib Nabi', 'Yudhvir Singh', 'Abid Mushtaq', 'Umran Malik', 'M Ashwin', 'Qamran Iqbal', 'Shubham Khajuria', 'Vivrant Sharma', 'Musaif Ajaz', 'Abdul Samad', 'Kanhaiya Wadhawan', 'Yawer Hassan', 'Lone Nasir Muzaffar', 'Kawal Preet Singh', 'Sunil Kumar', 'Vanshaj Sharma'], 'id': ['X0ihiiUJJcKzGAOiIybLCO1JQeiIJVe', '2016fcf0ce88944c11e69558028d29d', 'AeIhVQyXJiiibJ1iIceLJJzUO0KCGOi', '20182464a4dac6d211e8a30e02726df', 'OKiheAIOyQXLz1I0JJiicJCGiVUJeib', 'KcACOLJJbQiVeiihzi01JGJyOIUiXIe', 'bUOiQyIec0iiJVCJJihOLGXAJzeI1iK', 'eIVQh1iG0iyUcJAizJIXOKbeJiOJLiC', '2016c34d7e2eb6eb11e6842a028d29d', 'hicQGCIJAJiJiVKXbJLyU0Oii1OzIee', 'zJiei0ViQUKCJJGLIOXOJib1AiechyI', '201805b70bd4c6d311e8a30e02726df', 'JOGciAyXKiCeiI01IbLUJzOeJhiQVJi', '2017fc961001d41e11e792a5028d29d', '20222e4ce199454811ed9f33067d1ab', '2017ea701d6ed42511e792a5028d29d']}, 'Goa': {'name': ['Ishaan Gadekar', 'Arjun Tendulkar', 'Abhinav Tejrana', 'Suyash S Prabhudessai', 'Lalit Yadav', 'Darshan Misal', 'Rajasekhar Harikant', 'Vikash', 'Deepraj Gaonkar', 'Shubham Tari', 'Koushik V', 'Mohit Redkar', 'Heramb Parab', 'Kashyap Bakle'], 'id': ['hiQKeiOiIAJCJyVzbieXLiIJcO0U1JG', '0iGiOcIKzCh1AeIeJiJiVLUQibOJJXy', 'hOciQCJiGIJiIJKeUiO1JXyV0zLieAb', 'XeiiJJIzUyiJ0LAIOGcVQibhJieC1KO', '20173b1eadf9ab8e11e78020028d29d', 'KiiX0JIG1iIVAhcizJiCObeOLeJQJUy', 'JAicKbyJiULXeOhiJiGJCIzQI1O0iVe', '202245a582bea5d411ecb25d067d1ab', 'VIJhCK0JO1JOebciAeiLiJzUQyGXiIi', '20164d4ffe65b18711e6842a028d29d', 'JizLIiAQJhKyXO1ViUecbJ0JieGCiOI', 'iiiiJJJLyIUeCQOJX0heKbzAi1cOGVI', 'JJi0iG1AiOzQcChIiKXiUOeIVyLJbJe', 'y0LXI1IAOJiiVizeCceUOQihJJGbKiJ']}, 'Uttar Pradesh': {'name': ['Bhuvneshwar Kumar', 'Sunil Kumar', 'Karan Sharma', 'Shivam Mavi', 'Vipraj Nigam', 'Prashant Veer', 'Aryan Juyal', 'Priyam Garg', 'Sameer Rizvi', 'Rinku Singh', 'Siddharth Yadav', 'Abhishek Goswami', 'Madhav Kaushik', 'Aaradhya Yadav', 'Kartik Tyagi', 'Prince Yadav'], 'id': ['yJUcXziCJOGLbJeViK1IiihIQOJie0A', 'bIJJQhOiiUKOLCcXz1VAiGeyIiJ0eJi', 'JOhGeIyzJebiQiiJiXcK0J1CUAOLVIi', 'OiOi0yi1JIIzULbiJAeKCcQVXJhGiJe', '2021314c15ff1df411ec9eb102726df', '2022e1b87b493fe211ed9f33067d1ab', 'KiJiIiUe1QGchzOyJJCIXJL0bOiAVie', 'iVJIybeOJOILKzXceiCJi1JAii0hGQU', '20168fbd0eb6b07211e6842a028d29d', 'VIcLi1IJUAXzCeiiJQJKJebiyOi0GOh', '20169dff00dfb08c11e6842a028d29d', 'eUXyiAieILIJOiJK0COiQbciGJVhJz1', 'Lye1IAVUIbQiJJCOKc0GiziehJOiXJi', '201732230553d00f11e792a5028d29d', 'zJ0AJCOieVhyOXbiUQKJiGeiiIIJ1cL', '20215ee2dda245fa11ec9eb102726df']}, 'Himachal Pradesh': {'name': ['Innesh N Mahajan', 'A K Bains', 'Pukhraj Mann', 'E C Sen', 'N R Gangta', 'Mridul P Surroch', 'M J Dagar', 'Rahul R Chauhan', 'Vaibhav G Arora', 'Aryaman Dhaliwal', 'Arpit N Guleria', 'Divesh R Sharma', 'A P Vashisht', 'R I Thakur', 'Vipin Sharma', 'Amanpreet  V Singh', 'N K Kanwar'], 'id': ['2019776a0081e76b11e9a30e02726df', 'XiKJihCiLy1cI0iUIQiGbAOOJVeJJze', '2016ca1cb3a1afba11e6842a028d29d', 'h0I1bXKJViOJGiiCOUJIyzAJiieecQL', 'CVOiJiciXOJhQKLUI10AGybiiJIzJee', '201892ebef78d13211e8a30e02726df', 'AycJiKeJ1iIIXih0COObiJLziUQVGeJ', '2022240d0147a5d611ecb25d067d1ab', '20182b29a0f9ad4f11e89b5f028d29d', '20198b5be65ce34711e9a30e02726df', '2018fff979cff6ae11e79b5f028d29d', '202168ba606c421711ec9eb102726df', 'XiGLJUQiJVOhKyJiOIbe0i1IizCAeJc', 'ILiJJzciOVbCiKi1UQ0eJyXIAGOehJi', '2017d90d7543a8ec11e78020028d29d', '20179129226cd02e11e792a5028d29d', 'eJGiCieK0iiJ1OUyAhbViJzILOIJcQX']}, 'Punjab': {'name': ['Abhishek Sharma', 'Gurnoor Brar', 'Ashwani Kumar', 'Sanvir Singh', 'Ramandeep Singh', 'Harpreet Brar', 'Mayank Markande', 'Prabhsimran Singh', 'Anmolpreet Singh', 'Salil Arora', 'Nehal Wadhera', 'Naman Dhir', 'Ayush Goyal', 'Vishwa Pratap Singh', 'Gourav Choudhary', 'Harnoor Singh', 'Jass Inder', 'Raghu Sharma'], 'id': ['i0IJLJ1GJOycQAVzCeheiUbXiiIOKJi', '20196546ad16f49011e9a30e02726df', '2019b3f822e9e34811e9a30e02726df', 'OcbAiezOeIJLUiy1GQJi0iJCiVhJXIK', 'iXyOJLIQAbKiChG0JiVzIieiUJOceJ1', '2018566b707bdc2011e8a30e02726df', 'Q0GJOJCJLO1UiKiAeJIiyzVbXihcieI', 'yibzO1G0KiJeQAeicUIOCJJiihLJVIX', 'KzQJ1ehGACiO0cbOViiXiLIyJIiUJeJ', '2017ba78106faac911e78020028d29d', 'bUAIiLcJGJKXJiee1zy0iVOJOhiQCiI', 'beyJUOLGCiJ0iJii1ihKIIOzecVQJXA', '2017cee07574d1dc11e792a5028d29d', '20198f4819e0f49511e9a30e02726df', '2017ecb56520aac711e78020028d29d', '2017c5ea8b9fd1e011e792a5028d29d', 'IJyGKVCciiIzibOOAUeXJiJ1JQiLeh0', 'IiJiJVizGKyOeQchA01UeIiJiOCXLJb']}, 'Services': {'name': ['Gaurav Kochar', 'Harsh Wardhan', 'Abhishek', 'Mohit Ahlawat', 'Vineet Dhankhar', 'Arun Kumar', 'Nakul Sharma', 'Jayant Goyat', 'Pulkit Narang', 'M S Rathee', 'Vishal Gaur', 'V U Yadav', 'Mohit Jangra', 'Vikas Hathwala', 'Nitin Tanwar', 'Ravi Chauhan', 'Sandeep Nishad', 'Avinash', 'Kuwar Pathak'], 'id': ['JhJiXLOOUezQiGCcbVAiiJJy0I1KieI', '2021855a25661e9011ec9eb102726df', '201824dce68cbb5a11e8a30e02726df', 'hJOieJOJJyKVGCieAcXQib01iIzIiUL', '20194cdbed25e74511e9a30e02726df', '20174be47782ce9111e792a5028d29d', 'hJLiICKXIQeyJVJOibUiiicJezOGA10', '202268f3301940a411ed9f33067d1ab', 'XQhiKibiJiOA0zcJyL1iJOGJICVeUeI', 'JC1QbzihXLiJyJiOcOeVJIeGi0IUKAi', 'Ie0iiKVJLeiJXibhGAOCJUiQOcIy1zJ', 'IXzVJ1iicJbAiIeiKeQOOLJihJyGCU0', 'QiiiGAJb0hJJJzVyOiKIiCIO1ceXeUL', '0iOLICKiiyVee1iJJUhOizJAIJGXQbc', 'iCeJhJLGi1czV0iXiJJiQObIUOAKyIe', 'KGVehiIOLJ0JiXbCJAOe1iUyiJcIQiz', '201954784705ea9211e9a30e02726df', '2022e7a49f5d601d11ed9f33067d1ab', 'hObKCXi0yiUeiJcILV1JzeIGJQOiiJA']}, 'Gujarat': {'name': ['A Nagwaswalla', 'H V Patel', 'Hemang Patel', 'Rm Bishnoi', 'Vishal B Jayswal', 'Yash Doshi', 'Aarya Desai', 'Urvil Patel', 'Ripal Patel', 'S D Chauhan', 'Abhishek R Desai', 'Umang', 'J K Bhatt', 'Dhrushant H Soni', 'Rishi Patel', 'Hiten Mahera'], 'id': ['XiC1OiJbJIJQVezJUGIi0yLiKciheAO', 'J0ULeeiciJJX1bOiOCVAJizyhQIiGIK', 'hJJyeiiciJ1QeVLAbOOzCXiIKGUiIJ0', 'ieJhIbiGeXU0KCQLIV1iJAOiJOycJzi', 'CO0IiQJiiiGyJhXeUbIcVeAzJiOJK1L', '20180be0cbc0f5fb11e79b5f028d29d', '2017ec033fb9cf5511e792a5028d29d', 'XJiyOV0iAhbCGieIJKIQ1eiJUczOiJL', '2019b8964b02d90611e9a30e02726df', 'y0cIiiOLzAKGQOiibUVhiJeJJIe1JCX', 'iXiQJieUI01hGicCzAIObOiyJJKVeJL', 'XIO01JbiCUiiOJQAceKzeVGJJiiILyh', '20166d567a4aafe811e6842a028d29d', 'QOhVUzJiy1JIJJeCbKiicLGIOiXA0ie', '201834807cabc17011e8a30e02726df', 'A0VUKeJzOJGieiILCiiI1XhQcyJbiOJ']}, 'Tripura': {'name': ['Babul Dey', 'G H Vihari', 'Bikram Kumar Das', 'Sridam Paul', 'Sentu Sarkar', 'Vijay Shankar', 'M B Mura Singh', 'Swapnil K Singh', 'Saurabh Das', 'Amit Ali', 'Indrajit Debnath', 'S S Sutradhar', 'V K Saha', 'Sankar Paul', 'Saruk Hossain', 'Tejasvi Jaiswal', 'Arjun Debnath', 'C K Paul'], 'id': ['iJCiOU1GAKbQeiOhI0yIeiLVXJiJJzc', 'KbiyeJQIhLOJC1iUGzJiAOiJ0VXceiI', '0IiiiIcX1OheJUJiQKzVeyJAibJOLGC', '20165200c3d3b22f11e6842a028d29d', '20213aa632fe1ddf11ec9eb102726df', 'GJi0JzVeUiibyOK1IiQAOCicLhJeIXJ', 'GOiILzJyeXeIcJ1KAiiiJCVOJhiQUb0', 'O0UicGJiiCAVKJhbLiiJyXJIeeQ1zOI', 'zeXhbiJICOIViQieLUOyG0iJcJK1AiJ', '2016ef90286db21611e6842a028d29d', '2017a2b52142da6611e792a5028d29d', '1eVOOJCzIiciJJIJK0ebXGUAiiiLhQy', 'JziAXiheiCGKQiJJVUeyObLJIO0i1Ic', 'ieKyiJOAVeiibhQJL1OI0JcIUJCzXGi', 'eiLO0JOiJyUhIQzGXAeicC1ViKiIbJJ', '2023e614deacbdcc11edaab806e2a20', 'JiLIQihKeiiJ1VAXcJUybJeziI0GOCO', 'OXeILIJyJCeJKOV1iiJiQAzi0cbGhUi']}, 'Saurashtra': {'name': ['J Unadkat', 'Chetan Sakariya', 'Chirag Jani', 'D A Jadeja', 'Gajjar  Sammar', 'Krains Fuletra', 'Vishvarajsinh Jadeja', 'H Desai', 'Prerak Mankad', 'Jay Gohil', 'Ruchit Ahir', 'Luckyraj', 'P Rana', 'Siddhant Rana', 'Parth Bhut', 'Yuvraj Chudasama', 'Ankur Panwar'], 'id': ['iJy0OICJehGiUJVLiA1KIbziOeXQiJc', 'yK0eOAGiO1JVQIeIbULiCXhiiciJJzJ', 'eG0eCOQJVJIUKOLzcbIAJhJyiiiiiX1', 'yIKXeiOL0JAbQiiUJiJeGCVcO1izJhI', '2018f64d2959d36411e8a30e02726df', '20215a7c4e9e1b9c11ec9eb102726df', 'IJUAJiiK0hQIeXJy1OiVLOcJCibzeGi', 'UiKJViiJO1eIGAzeXQOLC0iJIcyJbhi', '0icKiJX1QObOihJiiAIyJeLJVeUICGz', '20170645e644a37c11e78020028d29d', '201932c9d568df5511e9a30e02726df', '2023816f6bf86e7611eeb03006e2a20', 'LcihiUOJiJKbiGeIyzi1AVX0CQeJJOI', 'cIbOJiUeiACeOXVJIyi0L1ihzJGJKQi', 'cieKehO0AJiyzbQCO1VGiiJIXILiJUJ', 'IzJJJVKOCeXGci0LIii1beAihJyiQUO', '201902745a5a15b411eaa30e02726df']}, 'Uttarakhand': {'name': ['Yuvraj Choudhary', 'Avneesh Sudha', 'P S Chopra', 'Aanjaneya Suryavanshi', 'Kunal Chandela', 'Shashwat Dangwal', 'Himanshu Bisht', 'Madhwal A', 'Rajan', 'Agrim Tiwari', 'Suchith J', 'A C Ayachi', 'Sanskar Rawat', 'D Negi', 'Bhupen Lalwani', 'Aarav Mahajan', 'Naveen Kumar Singh', 'Rahul Raj'], 'id': ['201715131504d1d411e792a5028d29d', '20162cec1489ad7411e6842a028d29d', 'eUJzhL0ICGbJyiieVcAQKJXiOi1IiOJ', '2017ff5f353fd00d11e792a5028d29d', '201629d269be913211e69558028d29d', '20188795e7f2d38a11e8a30e02726df', '2018941d3759e67a11e8a30e02726df', '201913f97eb2daa311e9a30e02726df', '2021f60219eb545411ec9eb102726df', 'iyJOJbIUJGzXIeJOhCAViiii0Ke1cLQ', 'OyXQiJKCizOiIJJcJGU0ihebIiLVA1e', 'VGbhJ0XyCieOJIJKUeiQiLAIcJ1ziiO', '2019097744b2e9b811e9a30e02726df', '20191409ec51daac11e9a30e02726df', 'IOi01XJcQiVCKLGzOJiUebJJIAyhiie', '2021c0db2add1dbb11ec9eb102726df', '2018c12acd00e67b11e8a30e02726df', '20255825ef94a7ea11f083bd061540e']}, 'Karnataka': {'name': ['Vidwath Kaverappa', 'Vidyadhar Patil', 'V Vyshak', 'Shubhang Hegde', 'Shreyas Gopal', 'Pravin Dubey', 'K L Shrijith', 'Mayank Agarawal', 'Karun Nair', 'Smaran R', 'Abhinav Manohar', 'Devdutt Padikkal', 'B R Sharath', 'Manvanth Kumar L', 'Macneil H N'], 'id': ['20173fea7256a82e11e78020028d29d', 'IiAVczIyUeiiKX1OJJLCibQOJGhiJe0', 'iJiiIiJUXeyVOJQecObiILz1AChGJK0', 'QIiVOiAXJLzyJiOJ1CUeJhGK0cIiibe', 'VAzG0QCKJihJJILeieiibi1cIyUOJXO', 'IJzJQJJ0biyiCecKiVUOi1hXOeIAGLi', 'eb0LIGUJiCJKQOJVyJiichOiziAeIX1', 'GJIOIyVUihiLiAzJXOeJ0KQebii1JCc', '0JchQiibeAGyCUiJeX1IKzLIOJiJOiV', '2017a60d333ad1be11e792a5028d29d', '2016c92165c08c9611e69558028d29d', 'iGeJOCJAQLeihUiJIXzi1bJ0VIiKcOy', 'XieI1hLIOKeUzbA0VOCJJiJiQGiciyJ', '2022efc80d17430d11ed9f33067d1ab', '2019c6867212e03911e9a30e02726df']}, 'Andhra': {'name': ['K S Bharat', 'Ashwin Hebbar', 'S K Rasheed', 'Avinash Pyla', 'Ricky Bhui', 'Saurabh Kumar', 'P Panduranga Raju', 'K V Sasikanth', 'Pvsn Raju', 'Prithvi Raj Yarra', 'Stephen', 'B Yeswanth', 'Sdnv Prasad', 'Tripurana Vijay', 'K Nithish Kumar Reddy', 'M Hemanth Reddy'], 'id': ['iiVJIJeU0LJXhy1GieiQJOCcIziKAOb', 'eAOiIizIQGOeVXihL1cCJiyJJiU0JKb', 'OiGOzQyJbViJCiIJUXAh0cKiJIeLie1', '20195a3dffecf71b11e9a30e02726df', 'iiLiJXQCizOJJIy1UAKIeOeibVG0hJc', 'iOKJJiXJGiheIibCOLVieUQz1AJcy0I', '2024287a1a20b84111ef83bd061540e', 'hOXAzOyce1iViJLQb0ieiIIJKJGJUCi', '2016b1ac746aa27711e69558028d29d', 'JJQ0JehC1UiLGJiyAieKIzXiciObIOV', 'GLiiCAJiViUXOeO1hicQIJzJ0IybKJe', '2022e96375e8455d11ed9f33067d1ab', '20192b103bd1e36711e9a30e02726df', '201768408556d4d611e792a5028d29d', '20161382502ab62911e6842a028d29d', '2018d72fc375d04511e8a30e02726df']}, 'Assam': {'name': ['Mukhtar Hussain', 'Mrinmoy Dutta', 'Akash Sengupta', 'Jitumoni Kalita', 'Avinav Choudhury', 'Sibsankar Roy', 'Pradyun Saikia', 'S C Ghadigaonkar', 'Nihar Deka', 'Riyan Parag', 'Saahil Jain', 'Bhargab Pratim Lahkar', 'Denish Das', 'Abdul Ajij Kuraishi', 'Sadak Hussain', 'Ayushman Malakar', 'Rohit Sen'], 'id': ['K0LIAVihiyGeUizJ1JiJXQOICObcJei', 'chGXeJJiOUJeJQii1iIIOiLAK0CVyzb', 'IJOeQ01iCeyiXiziVLAKcJJIGOibhUJ', 'JCJeiJizeVIIhQbXiKOiJGiALOcy0U1', 'eJJhIAiGOLVOcy1iJeU0JCiQIzKXiib', 'CGIcQiziJiebiVAyieLXUI0JOKOh1JJ', '202237c23514454611ed9f33067d1ab', 'iJIbGJyiiUeQIeLi0CVJzOcihJX1KOA', 'AOICybiJKiXJVzJLiiUJQOIe0e1hiGc', 'JeicCLIGAXyheJ1UJbJIO0QVizOiiKi', 'UizyiCiVXeiQLcGJJIJ1O0JhIOKAebi', '2021f4eb79d14d1111ec9eb102726df', 'X1eOUbI0iJcLizCieIJQVJGiJKAOiyh', 'iiJJiGcOAJeQb1KLXOVUih0IzJeIiyC', '2024d2e535cca12611ef83bd061540e', '202100cdb0dc670611ec9eb102726df', '202274fb54bf5a9311ed9f33067d1ab']}, 'Odisha': {'name': ['Swastik Samal', 'Gaurav Choudhary', 'Subhransu Senapati', 'Biplab Samantaray', 'Sambit S Baral', 'Prayash Kumar Singh', 'Sourav K Gouda', 'Rajesh Mohanty', 'Vageesh Sharma', 'Pappu Roy', 'Badal', 'Govinda Poddar', 'Subham Satrujit', 'Aditya Rout', 'Sandeep Pattanaik', 'Soumya Ranjan Lenka', 'Aashirwad Swain', 'Anil Parida', 'Sarbeswar Mohanty'], 'id': ['cUJIieGiiChOAbzVIX10iQOJiyKLeJJ', '20206255b3864b4e11eb993e02726df', 'KciiiyhJJe1ILJVGAIbJiQzOXU0ieOC', '1JJiXzUOiOJJQceiiehLiGbKyIIAVC0', '2017cc1ac9e2d05111e792a5028d29d', '20167e1afe9c8bf311e69558028d29d', 'bKJhOiOUCJIzVciJiQ0IiAJLGeX1iey', 'OiCi0iQVKiJeLJyI1AOXJJchIUbzieG', 'ehKIiiGicVeJLUIOyJ0J1OiCziJAQXb', 'K0eiJeXizGVIOiAchIJiJyQ1OLUJCib', '20232905dd426dbe11eeb03006e2a20', 'iyJKebAeO1hcCUiGIiIiJ0iOLVJQJzX', '2018a167b599c6d511e8a30e02726df', '2016324322c7b3c511e6842a028d29d', 'QJUeiGJVLbeOIXy1IOCchiAJiiizJ0K', '202582644184a50811f083bd061540e', '2018aeb37bddd36111e8a30e02726df', '201802ed8278c6d411e8a30e02726df', 'eUzyOiJGCQ1cIJOiKiLiebiV0XhIJJA']}, 'Kerala': {'name': ['Saly Viswanadh', 'Nidheesh M D', 'Asif K M', 'Ankit Sharma', 'Abdul Bazith P A', 'Akhil Scaria', 'Sanju Samson', 'Rohan S Kunnummal', 'Ahammed Imran', 'Salman Nizar', 'Vishnu Vinod', 'Sharafuddeen N M', 'Vignesh Puthur', 'Mohammed Azharuddeen', 'Krishna Prasad', 'Biju Narayanan N'], 'id': ['JJ0iLbieQOJAhOCUXy1eJcIiVzGKiiI', 'AL1eJIiOcyJJehCbi0IzViGUJOiQXiK', 'JiO1eJi0ALJViICJhziKXiQIyGcOebU', 'ciiJbIeiUeJAQXJyihi01LKJzCOIVGO', 'QXezUiiKeGhL0VyCOOAJJicbJI1JiIi', 'VJQchiOiiCIAIJJKyUXi0ObGLieJ1ze', 'iJUViQ01GecizhCJIAbiyXJOiLIOJeK', 'XJz1AcLJeKIOhbCVGQJi0iIeOiUJiyi', '202118653d6663bf11ec9eb102726df', 'eVi1OJe0iJcKiOiAIJXQJULCbyihGIz', 'iyKLicQiOJIbJ0iVie1AeUGJXhIOJCz', '20211131f4d9396611ec9eb102726df', 'eUV1KIIAiJObQhGizeyiJCOJiLJX0ic', 'eIiGOUeiyiIcOCJLVhbJ0JXiKiJQAz1', '1OhziJiJCIyiecLJJ0AeKiQXUVIiGbO', '2024fa4c8cbc860511ef99b1061540e']}, 'Bihar': {'name': ['Vaibhav Suryavanshi', 'S Gani', 'Ayush Loharuka', 'Atul Prakash', 'Bipin Saurabh', 'Akash Raj', 'Suraj Kashyap', 'Khalid', 'Bhanu', 'Malay Raj', 'Md Izhar', 'Nawaz', 'Piyush Kumar Singh', 'Sakib Hussain', 'Pratap', 'Mahrour', 'Himanshu Singh', 'Amod Yadav', 'Sachin Kumar Singh', 'Kundan Verma'], 'id': ['201998d319c4ea7611e9a30e02726df', 'KVzJhLeOicCiXJiGiAJQOyJIUi01eIb', '20216efc21781c3c11ec9eb102726df', '1eyUii0JKIOIJCVieLGXcJhQbiJzAOi', '20183b53da19bfcb11e8a30e02726df', 'UiXJe1JhLeJCbiiQGVIiAOJ0KOzcIyi', '2017f7db9c15cdf511e792a5028d29d', '2019f162759dfa4311e9a30e02726df', '20189727e2ccfd4a11e79b5f028d29d', 'KXiLAiybQJi0IGheOIJUiVJzeiJ1OCc', '2021278020674e8c11ec9eb102726df', '20188f6c32f3fa8e11e79b5f028d29d', 'G1JiiJOzQehcIi0JiLbIVXUKCAeiyOJ', '20217caf5f041c5911ec9eb102726df', '20190db52d430b6211eaa30e02726df', '2018d91be9fcd75211e8a30e02726df', 'yiKCIJQeiihJ0AOOUzG1JLiIbJeVXci', '2018707f8718bfc511e8a30e02726df', '201814992c67dc3c11e8a30e02726df', '2025cf57b2e1a81911f083bd061540e']}, 'Chandigarh': {'name': ['Sandeep Sharma', 'Jagjit Singh Sandhu', 'Rajangad Bawa', 'Rahul Singh', 'C Dhindsa', 'Raman Bishnoi', 'Manan Vohra', 'Arjun Azad', 'Shivam Bhambri', 'Gaurav Puri', 'Arjit Singh', 'Nikhil Sharma', 'Bhagmender Lather', 'Nehal Pajni', 'Nikhil Thakur', 'Amrit Lal Lubana', 'Sanyam Saini', 'Rohit Dhanda'], 'id': ['i1QhcIiJJXbVOyCieKe0JIJOiiULGAz', 'IbhiViQUeJLCiOeA0OGK1JJJyiIcXiz', '2018fc2a2d5bc55c11e8a30e02726df', 'eKhGUA0OIzXLiICciOiV1iyieJJbJQJ', '2019ffc9d93ee11d11e9a30e02726df', 'VIibAiJyJiCJiOOUJQLXc0KI1ieeGhz', '0yiIKLIiOJVeiUzQCAiJOehJcXiJG1b', '201641c0d23fae5311e6842a028d29d', '0iiIIO1GbJCJOeAiiXLhyVJzcUKieQJ', 'OeJLIziy0UJIiihJKCbJ1ciAXGOiQVe', 'QIVJiiiLiJ0hXeJGbICJA1KOcezyOiU', '20195287b1f4f55e11e9a30e02726df', 'OLVUhb0GyKOCQIzciJAXJeJeiiJi1iI', '20164c73dd10ad9311e6842a028d29d', 'iyeOJLIcCJhJiVibiAeJ1Xi0QGUKzIO', '2019c8f89c87f55811e9a30e02726df', '20171acb79d5c0ac11e792a5028d29d', '2019ebeaad15deb011e9a30e02726df']}, 'Madhya Pradesh': {'name': ['Ankush Singh', 'Abhishek Pathak', 'Harsh Gawali', 'Harpreet Singh Bhatia', 'Aniket Verma', 'Venkatesh Iyer', 'Rahul Batham', 'Shivang Kumar', 'Mohd Arshad Khan', 'Tripuresh Singh', 'Shivam Shukla', 'Kumar Kartikeya Singh', 'Rajat Patidar', 'Rishabh Chouhan', 'Mangesh Yadav'], 'id': ['yKJOiAi0GV1LceiOihXCbQUIeIJiJJz', 'QJieyJKzCJiJii01cObIeLVXGUhOiAI', 'eLbi1JheOOiCIQXAIi0iicKJzJyUVJG', 'UiOJJK0yiiiCVeJ1eOXihIIJQGLbzcA', '20185ad57de6c54b11e8a30e02726df', 'eic0LJzOOIe1IQAiXJCiUGibJyiVhKJ', 'zeJUQiIOGyicJihKJVOAbXieLiC10JI', '2017cc61c826d11411e792a5028d29d', 'JiiVGLhzIAeUe1ciiXQJCiO0KIOJJyb', '2016d2d88846b09111e6842a028d29d', '202405dc319c86c211ef99b1061540e', '2017b97898cda37611e78020028d29d', 'KhJXQJGIiOi0eUzLIiCVbJJe1ciyiAO', 'iOhiGXJbJieLVQJeICJzO0iiAyK1IcU', '202360e4e6b96b3a11eeb03006e2a20']}, 'Hyderabad': {'name': ['Ctl Rakshann', 'C V Milind', 'Tanay Thyagarajan', 'Ajay Dev Goud', 'Md Arfaz', 'Ashish Srivastav', 'Tanmay Agarwal', 'Aman Rao', 'Pragnay Reddy', 'Buddhi Rahul', 'Bhavesh Seth', 'K Nitesh Reddy', 'Md Siraj', 'N Nitin Sai Yadav', 'H K Simha', 'Rishiket Sisodia', 'Mickil Jaiswal'], 'id': ['IJbi0eJOhiGJXAJQzCecU1LKiOiIViy', 'UAOcJbCeGzXiVeIi0LOQJihI1iJJKyi', 'iGJVOUKyiJhJbJ01IieXOiAcQizeILC', 'ceOKUIibOiVJQ1XCLyihAJJziGiI0Je', '2017b3b382e8d10311e792a5028d29d', '201895148144c22111e8a30e02726df', 'c0JJeGViKLQJUOheiIXz1iOiAiIJbyC', '2017276eb9e6cdbb11e792a5028d29d', 'IyiUQJViJJJeIhc1CXz0GLKbOiOAeii', 'iJhy1zQGOibIiecXKOUAJii0eJVICLJ', 'yOCXcIGJJ10VihLJbKAzeJIQiiiUeiO', 'JAIObViUJezh01XiyLOiiJJciKQGeIC', 'UiLzXQiGJbJ1IOAy0cJehOJeiCKIVii', '20188339aee3d39211e8a30e02726df', 'yKzAQIJUJcOCOh0i1iJVbIeGJiXiLei', '20163d5feeb7afe611e6842a028d29d', 'Ji1JbihVCIiGOi0AeQKILiJJXycezOU']}, 'Baroda': {'name': ['Shashwat Rawat', 'Vishnu Solanki', 'Shivalik Sharma', 'K H Pandya', 'J M Sharma', 'A Sheth', 'Bhanu Pania', 'N A Rathva', 'Rasikh Salam', 'Raj Limbani', 'Mahesh Pithiya', 'H H Pandya', 'Ah Pasi', 'Dhruv Patel', 'S D Patel', 'Chintal Gandhi'], 'id': ['20181f4f11a3c0b611e8a30e02726df', 'zLhCiIO0JIUVQAJ1GeiiiyXeKOJJibc', 'Q10yJUizOeGbiVLOKIIJXeJchJCAiii', 'UicQOhOJ1XACJeGKyeVJbiLIiJiiIz0', 'XGiIhViyKiIAOLJU0OCe1ecJJQzJibi', 'IOJUiIKe0zb1XVeOiiGiicJALJJQhyC', '201959c3f55cd9e111e9a30e02726df', 'iIX0QAiJeez1GcKbIJVOyiLiCUJJhiO', '2018d1909e70bbd611e8a30e02726df', '202142403be6186611ec9eb102726df', '2019cad5abe0e10511e9a30e02726df', '1hicJieAIiGK0eiiJCVbOXyQLOIJUzJ', '2017128cc768c61c11e792a5028d29d', 'CbQOiyGKAIVO1iiihJeJXIeJ0ULizcJ', '20178b406f1da8dd11e78020028d29d', '202121f5a6974e5211eb993e02726df']}, 'Bengal': {'name': ['Mohammed Shami', 'Saksham Chaudhary', 'Writtick Bijoy Chatterjee', 'Sayan Ghosh', 'Shahbaz', 'Pradipta Pramanik', 'Abishek Porel', 'Karan Lal', 'Shakir Habib Gandhi', 'A R Easwaran', 'Sudip Kumar Gharami', 'Akash Deep', 'Mukesh Kumar', 'Keswani'], 'id': ['ChbiJ1JcJIiiVXKQeyOAeGUiiOL0IJz', '2022ef4f5d3160c911ed9f33067d1ab', 'ViiAJzJLhI1OGeKbIyQiUJ0cJiXeCOi', 'XIOVychi0iebieJCOUJQiJJzKLIiAG1', 'ecQ0AiG1eXJOzOCJLJVyhJiUKbIiIii', 'yzbhiALOJeIUi1I0iQOcGViiJCKJXeJ', '2016aa74c1eab2f911e6842a028d29d', '2016ae2005eab30c11e6842a028d29d', 'AhGJQIJc0KJIOVCJUXiiyeiiL1ibeOz', 'Ji1iGzUiOCKieeLAXQJhIIiJV0byJOc', 'CK0LibGi1IXJJiViOUieQJOJcIyhzeA', '2018d8851d0bdd9511e8a30e02726df', 'AJCi0JJVJeXhyUziiOLIeiIicbGKO1Q', '2019f1986c36df5811e9a30e02726df']}, 'Pondicherry': {'name': ['V A Bhardwaj', 'Ajay Rohera', 'Akash Pugazhanthi', 'Parameeshwaran S', 'Aman Khan', 'Krishna', 'J J Yadav', 'Sidak Singh', 'Himanshu Sharma', 'Adil', 'Aravind', 'Raghav Goyal', 'Gourav Yadav', 'Anand Singh Bais', 'Puneet Datey', 'Aditya N Garhwal', 'Shrikaran', 'Sagar P Udeshi', 'Vijay R', 'Bhanu A Anand', 'Ragavan', 'R Jashwanth Shreeram', 'P Thamaraikannan', 'Vikneshwaran Marimuthu', 'Md Shafeeq', 'Sivamurugan M'], 'id': ['IzOcKC1hXVeJi0QJiLOIJiGyeibJAiU', 'V1KiJOGiJJzhJby0eiIiAOUQLXceCIi', '20183b6eabe7d1df11e8a30e02726df', '202266cd181c43ae11ed9f33067d1ab', 'O0iJQJKihGXVJLii1JCbIAIzeycOUie', '20220619dbd443ad11ed9f33067d1ab', 'iOVOzKiiICIiUQeeGLXJb10JcJiJhAy', 'ihGJKJJiX1J0UeOibiiIzVCOLIeyQAc', '2023fc56a59c69b211eeb03006e2a20', '20235570ffc973e711eeb03006e2a20', '2018557c38a9d91611e8a30e02726df', '20168e66499dae2111e6842a028d29d', 'JiiIecy1JUKXiCiGAILQhebJOiJV0Oz', 'eXzeJJ01iVCKGcOOAiJILiUJQiiybIh', 'Qiee0KIOLibAJizVOUhJX1JCciJGIyi', 'IiiUy0CAezOiiJ1LJeXbicQIKhOJVJG', '2025ca502e17c93211f083bd061540e', '20183a59ab35b7f411e8a30e02726df', '20205f5a1fda45d711eb993e02726df', 'iJcOVJJAi0LiJebGiOe1XhQKziyCIIU', '20216b5e270d688811ec9eb102726df', '20190cd4c2b2e69711e9a30e02726df', '20192b35ab382e0e11e9a30e02726df', '201864437e6eb80311e8a30e02726df', '2022cf3007eda5d211ecb25d067d1ab', '2021a4e2587f445411ec9eb102726df']}, 'Haryana': {'name': ['Anshul Kamboj', 'Anuj Thakral', 'Nishant Sindhu', 'S P Kumar', 'Yuzvendra Chahal', 'Samant Jakhar', 'Parth Vats', 'Arsh Ranga', 'Ankit Kumar', 'Yashvardhan Dalal', 'Vivek Kumar', 'A R Rana', 'Bhuwan Rohilla', 'Ashish Siwach', 'Ishant Bhardwaj'], 'id': ['IKzbGiLiQceU01AVXJiieJChOiJyOJI', '2018c01215c0c94711e8a30e02726df', '201625cf5617ae1d11e6842a028d29d', 'XJ1IOeIcVCibQOJiieJihALUGJKyzi0', 'iiziL0iOie1JbKeJVIGcCJXIyUhAJQO', '20182701f07ed44111e8a30e02726df', '2017740469fdcea711e792a5028d29d', '201919e7ad67e74f11e9a30e02726df', 'KyIJUVeiJiXbeOiOCicJAQJiz1GI0hL', '2022ecb0958740b711ed9f33067d1ab', '2018bc42e49cc94f11e8a30e02726df', 'JL1iOeJ0UVbiiJciAKzGXIIJQeCyOhi', '2016911bacfaad7a11e6842a028d29d', '202128f2e49b5f1311ec9eb102726df', '2021bee0ac451ad411ec9eb102726df']}, 'Delhi': {'name': ['Priyansh Arya', 'Yash Dhull', 'Ayush Badoni', 'Nitish Rana', 'Anuj Rawat', 'Himmat Singh', 'Mayank Rawat', 'Simarjeet Singh', 'Prince Yadav', 'Suyash Sharma', 'Ishant Sharma', 'Tejasvi', 'Digvesh', 'Money Grewal', 'Navdeep Saini', 'Ayush Doseja', 'Harsh Tyagi'], 'id': ['2017b5036d9dc05e11e792a5028d29d', '2017398ee153d51911e792a5028d29d', 'iUJeXIbiz1VGLe0iiJcIOJQCAKJhiOy', 'hVUcKiLiyiOiXbJIJ0JJCee1OGAizQI', 'ACKIeUOzI0iJihebicGLQyXJJ1VOJii', 'c01IiUQOJiiAiCKzeGXJyVOIheJJibL', 'OiAIcXOJeCGLyUhJi1JQ0iiziIbVKeJ', 'y0ieiOOUhAJQiVi1IJJJzCKGcbXLiIe', '2018137872bbea5211e8a30e02726df', '2022c4a92d95658a11edaab806e2a20', 'UiOeJizJOeGXJiVybJ0iiIK1CIAhcQL', '20171d028d7bdcdf11e792a5028d29d', '2019e3c7107b122b11e9a30e02726df', '20195ea81c4c0b5411eaa30e02726df', 'iVieOCQ1ibOJ0cyAJizeKJIhXUJILiG', '2019a78369c70ac511eaa30e02726df', 'iVb1eKzUhXiiJOiL0iecJCQJGyIAJOI']}, 'Jharkhand': {'name': ['Vikash Singh', 'Saurabh', 'Sushant Mishra', 'Bal Krishna', 'Utkarsh Singh', 'Anukul Roy', 'Ishan', 'Kumar Kushagra', 'Virat Singh', 'Pankaj Kumar', 'Robin Minz', 'Rajan Deep', 'Amit Kumar'], 'id': ['eJJiOiCyiQVzXUAKIhJIObi0cL1iGeJ', '20194120d99fcfc611e9a30e02726df', 'SEA8qDc70eYxFUBotrI5xu1M9j103uQ', 'iiAGhcIi0J1QJCLiiyOzeXKJebVJUIO', 'iJAO1yOiUVCiiXJce0hKbJGLeQizJII', 'izXKeJhJIUbiOVQLeGAy1CicOi0JIiJ', '1KVLIceOQbJhCIiXJJiiUOizAiGJe0y', 'ciQKXeIhzieJGO0VIAiJ1LCOJUyiibJ', 'JziLeIGbJViI01CcOihJJeyXiAKQUOi', 'eGIVOcQiIi1eKzLXOJJUiyJibJ0ChAi', '20212cde2c361ad711ec9eb102726df', '2018eb9e5b1bd1df11e8a30e02726df', '2016e544157cad9011e6842a028d29d']}, 'Tamil Nadu': {'name': ['V P Amith Sathvik', 'Tushar Raheja', 'Shivam Singh', 'N Jagadeesan', 'R Sai Kishore', 'Shahrukh Khan', 'Rajkumar Ravichandiran', 'R Sonu Yadav', 'Varun', 'Gurjapneet', 'T Natarajan', 'B Sai Sudharsan', 'M Siddharth', 'S Mohamed Ali', 'Sunny', 'Ragupathy Silambarasan', 'Esakkimuthu A', 'S Rithik Easwaran'], 'id': ['201617e7ae6b9f6011e69558028d29d', '201609f59e8b9f4211e69558028d29d', '2024761f9697a11d11ef83bd061540e', '1IOQicL0zIiiJJGbUhOAiCyJXJeieKV', '1QzyJJeUKAiJGLCOVXiiIcbO0iIiheJ', 'Ch0JeOXVeyGbiiKJJUicIQizO1AiIJL', '2017ef93c71ae48711e6842a028d29d', 'iAiOJi1QVLyIJKCIbUcihzXieJeGOJ0', '2018ae1c7bafbb1911e8a30e02726df', '20216dcce81c420c11ec9eb102726df', 'eCUiIGI1ybzeOiJVOAiQLJXJic0hKJi', '20167c1304259cec11e69558028d29d', 'iC0KiUVbGOyzOJLiiAJIIJe1QeJhcXi', '2019279e4d98e43511e9a30e02726df', '202238626b6563f811ed9f33067d1ab', '20182a8eac6cbb1b11e8a30e02726df', '20257c68000ca41111f083bd061540e', '20165da31cc9a1c411e69558028d29d']}, 'Rajasthan': {'name': ['Mj Suthar', 'A M Singh', 'A Sharma', 'Kamlesh Nagarkoti', 'R D Chahar', 'B S Sharma', 'Shubham Garhwal', 'Deepak Jagbir Hooda', 'M K Lomror', 'Kartik', 'K S Rathore', 'S S Dhiwan', 'Karan Lamba', 'M Choudhary', 'A B Kookna', 'R B Chauhan', 'Rs Golada', 'Hemant Kumar'], 'id': ['eiOhCJGAziQJiU1OcK0IJLbeViyIJXi', '2017a66731ead35311e792a5028d29d', '20193d6ffff6e41011e9a30e02726df', 'iOiyQcieUO0XIiCK1zJJJiLJAGhVeIb', 'C1eiJOJOAViGiXcJJhiUIQIzKiL0yeb', '20192d407aed2ddf11e9a30e02726df', '2021d687a59b3b1a11ec9eb102726df', 'zeiQJI0C1KJiIbiJiULcyhAOVeXOGJi', 'byUJIVKXAei0ziJJcChQiG1OiILiOJe', '201920c31fe1ea4f11e9a30e02726df', '20192b18b5f5e41511e9a30e02726df', 'IO0iQOehKbVJGiyLIUzAieJiJXC1ciJ', '2016f15e019db60511e6842a028d29d', '2018b7391ae4d2c711e8a30e02726df', 'eiC1LheJizIKQi0ViOJybAIOiXUJJGc', '1Jyi0hKCILXiUGiOcOeJeAzbJVIiJiQ', '2017d5dd55b1a83011e78020028d29d', '20239813f75469b111eeb03006e2a20']}}

cricbuzz_squads = {'Mumbai': {'name': ['Ajinkya Rahane', 'Ayush Mhatre', 'Suryakumar Yadav', 'Siddhesh Lad', 'Sarfaraz Khan', 'Shivam Dube', 'Musheer Khan', 'Suryansh Shedge', 'Shams Mulani', 'Sairaj Patil', 'Shardul Thakur', 'Atharva Ankolekar', 'Tanush Kotian', 'Hardik Tamore', 'Tushar Deshpande'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'Bowler']}, 'Railways': {'name': ['Shubham Chaubey', 'Kunal Yadav', 'Mohammad Saif', 'Akash Pandey', 'Akshat R Pandey', 'Rahul Sharma', 'Navneet Virk', 'Ashutosh Sharma', 'Atal Bihari Rai', 'Shivam Chaudhary', 'Suraj Ahuja', 'Ravi Singh', 'Kush Marathe', 'Karn Sharma', 'Raj Choudhary'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', '']}, 'Kerala': {'name': ['Rohan Kunnummal', 'Salman Nizar', 'Krishna Prasad', 'Akhil Scaria', 'Muhammed Sharafuddeen', 'Ahammed Imran', 'Abdul Basith', 'Sanju Samson', 'Mohammed Azharuddeen', 'Vishnu Vinod', 'Krishna Devan', 'Ankit Sharma', 'KM Asif', 'Vignesh Puthur', 'MD Nidheesh', 'Sibin Gireesh', 'Saly Samson'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Odisha': {'name': ['Aasirwad Swain', 'Swastik Samal', 'Sandeep Pattnaik', 'Sarbeswar Mohanty', 'Anil Parida', 'Aditya Rout', 'Biplab Samantray', 'Subhranshu Senapati', 'Prayash Singh', 'Govinda Poddar', 'Sambit S Baral', 'Gaurav Choudhary', 'Sourav K Gouda', 'Subham Satrujit', 'Rajesh Mohanty', 'Pappu Roy', 'Badal Biswal'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Andhra': {'name': ['Ricky Bhui', 'Ashwin Hebbar', 'Pyla Avinash', 'Shaik Rasheed', 'Saurabh Kumar', 'Tripurana Vijay', 'K P Sai Rahul', 'Bhupathiraju Munish Varma', 'Srikar Bharat', 'Penmetsa Panduranga Raju', 'Prithvi Raj Yarra', 'Satyanarayana Raju', 'KV Sasikanth', 'Bailapudi Yeswanth', 'Cheepurapalli Stephen'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Assam': {'name': ['Sibsankar Roy', 'Saahil Jain', 'Jitumoni Kalita', 'Akash Sengupta', 'Riyan Parag', 'Pradyun Saikia', 'Bhargab Lahkar', 'Nihar Deka', 'Sumit Ghadigaonkar', 'Avinov Choudhury', 'Denish Das', 'Mrinmoy Dutta', 'Mukhtar Hussain', 'Abdul Ajij Kuraishi'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', '']}, 'Chhattisgarh': {'name': ['Amandeep Khare', 'Shashank Chandrakar', 'Sanidhya Hurkat', 'Sourabh Majumdar', 'Mayank Yadav', 'Anand Rao', 'Prateek Yadav', 'Shashank Singh', 'Ajay Jadav Mandal', 'Gagandeep Singh', 'Mayank Verma', 'Aditya Sarwate', 'Shubham Agarwal', 'Amit Kumar Yadav', 'Ravi Kiran', 'Ayush Pandey'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', '']}, 'Vidarbha': {'name': ['Aman Mokhade', 'Yash Rathod', 'Dhruv Shorey', 'Atharva Taide', 'Nachiket Bhute', 'Yash Kadam', 'Adhyayan Daga', 'Harsh Dubey', 'Darshan Nalkande', 'Varun Bisht', 'Akshay Wadkar', 'Shivam Deshmukh', 'Parth Rekhade', 'Yash Thakur', 'Praful Hinge', 'Umesh Yadav', 'Dipesh Parwani'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Hyderabad': {'name': ['Tanmay Agarwal', 'Bhavesh Seth', 'K Nitesh Reddy', 'Pragnay Reddy', 'Rahul Buddhi', 'Tanay Thyagarajan', 'Chama V Milind', 'Chinntla Rakshan Readdi', 'Ajay Dev Goud', 'Aman Akram Rao', 'Nitin Sai Yadav', 'HK Simha', 'Rishiket Sisodia', 'Ashish Srivastav', 'Arfaz Ahmed Mohammad'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Bowling Allrounder', 'Bowler', 'Bowler', 'Bowler', '', '', '', '', '', '']}, 'Madhya Pradesh': {'name': ['Harpreet Singh Bhatia', 'Aniket Verma', 'Akshat Raghuwanshi', 'Ankush Singh', 'Rishabh Chauhan', 'Saransh Jain', 'Arshad Khan', 'Rahul Batham', 'Ritik Tada', 'Shivang Kumar', 'Harsh Gawali', 'Abhishek Pathak', 'Kumar Kartikeya', 'Kuldeep Sen', 'Shivam Shukla'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler']}, 'Bihar': {'name': ['Amod Yadav', 'Vaibhav Sooryavanshi', 'Piyush Singh', 'Sakibul Gani', 'Suraj Kashyap', 'Khalid Alam', 'Mangal Mahrour', 'Akash Raj', 'Malay Raj', 'Bipin Saurabh', 'Ayush Loharuka', 'Nawaz Khan'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'WK-Batsman', 'WK-Batsman', 'Bowler']}, 'Chandigarh': {'name': ['Jagjit Singh', 'Gaurav Puri', 'Shivam Bhambri', 'Manan Vohra', 'Nehal Pajni', 'Nikhil Thakur', 'Arjun Azad', 'Raman Bishnoi', 'Bhagmender Lather', 'Arjit Singh Pannu', 'Nikhil Sharma', 'Rahul Singh', 'Chiragvir Dhindsa', 'Sandeep Sharma'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Jammu and Kashmir': {'name': ['Qamran Iqbal', 'Shubham Khajuria', 'Musaif Ajaz', 'Yawer Hassan', 'Vivrant Sharma', 'Abdul Samad', 'Lone Nasir Muzaffar', 'Abid Mushtaq', 'Auqib Nabi Dar', 'Kanhaiya Wadhawan', 'Vanshaj Sharma', 'Yudhvir Singh Charak', 'Umran Malik', 'Sunil Kumar', 'Murugan Ashwin'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Maharashtra': {'name': ['Prithvi Shaw', 'Ruturaj Gaikwad', 'Rahul Tripathi', 'Tanay Sanghvi ', 'Mandar Bhandari', 'Ranjeet Nikam', 'Arshin Kulkarni', 'Ramakrishna Ghosh', 'Rajvardhan Hangargekar', 'Niraj Joshi', 'Yogesh Dongre', 'Nikhil Naik', 'Azim Kazi', 'Vicky Ostwal', 'Mukesh Choudhary', 'Prashant Solanki'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Goa': {'name': ['Deepraj Gaonkar', 'Snehal Kauthankar', 'Darshan Misal', 'Kashyap Bakhale', 'Ishaan Gadekar', 'Heramb Parab', 'Mohit Redkar', 'Suyash Prabhudessai', 'Lalit Yadav', 'Arjun Tendulkar', 'Abhinav Tejrana', 'Rajashekhar Harikant', 'Felix Alemao', 'Vasuki Koushik', 'Amulya Pandrekar', 'Vikash Singh', 'Shubham Tari'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', '']}, 'Uttar Pradesh': {'name': ['Aaradhya Yadav', 'Rinku Singh', 'Madhav Kaushik', 'Priyam Garg', 'Abhishek Goswami', 'Sameer Rizvi', 'Karan Sharma', 'Prashant Veer', 'Kunal Tyagi', 'Vipraj Nigam', 'Siddarth Yadav', 'Aaditya Sharma', 'Aryan Juyal', 'Vineet Panwar', 'Sunil Kumar', 'Vaibhav Chaudhary', 'Zeeshan Ansari', 'Shiva Singh', 'Bhuvneshwar Kumar', 'Kartik Tyagi', 'Shivam Mavi'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Himachal Pradesh': {'name': ['Ekant Sen', 'Nikhil Gangta', 'Naveen Kanwar', 'Pukhraj Mann', 'RI Thakur', 'Vipin Sharma', 'Mayank Dagar', 'Aryaman Singh Dhaliwal', 'Ankush Bains', 'Innesh Mahajan', 'Vaibhav Arora', 'Arpit Guleria', 'Divesh Sharma', 'Mridul Surroch'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', '']}, 'Punjab': {'name': ['Vishwanath Singh', 'Nehal Wadhera', 'Naman Dhir', 'Anmolpreet Singh', 'Sanvir Singh', 'Abhishek Sharma', 'Uday Saharan', 'Ramandeep Singh', 'Salil Arora', 'Prabhsimran Singh', 'Mayank Markande', 'Harpreet Brar', 'Gurnoor Brar', 'Ashwani Kumar', 'Arshdeep Singh'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Gujarat': {'name': ['Hemang Patel', 'Vishal Jayswal', 'Amit Desai', 'Aarya Desai', 'Hiten Mahera', 'Ripal Patel', 'Umang Kumar', 'Yash Doshi', 'Jay Malusare', 'Japagnya Bhatt', 'Urvil Patel', 'Abhishek Desai', 'Saurav Chauhan', 'Harshal Patel', 'Ravi Bishnoi', 'Arzan Nagwaswalla'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler']}, 'Services': {'name': ['Gaurav Kochar', 'Nakul Sharma', 'Nitin Tanwar', 'Vikas Umesh Yadav', 'Mohit Rathee', 'Vikas Hathwala', 'Arun Kumar', 'Mohit Ahlawat', 'Jayant Goyat', 'Mohit Jangra', 'Pulkit Narang', 'Vishal Gaur', 'Vineet Dhankhar', 'Harsh Vardhan'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', '', '']}, 'Baroda': {'name': ['Shashwat Rawat', 'Dhruv Patel', 'Bhanu Pania', 'Ninad Ashvinkumar Rathva', 'Hardik Pandya', 'Krunal Pandya', 'Shivalik Sharma', 'Mahesh Pithiya', 'Raj Limbani', 'Atit Sheth', 'Vishnu Solanki', 'Jitesh Sharma', 'Mitesh Patel', 'Safvan Patel', 'Chintal Gandhi', 'Rasikh Salam Dar', 'Lakshit Toksiya', 'Amit Pasi'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', '', '']}, 'Bengal': {'name': ['Karan Lal', 'Shakir Habib Gandhi', 'Sudip Kumar Gharami', 'Shreyan Chakraborty', 'Abhimanyu Easwaran', 'Writtick Chatterjee', 'Sakshaim Chaudhary', 'Shahbaz Ahmed', 'Priyanshu Srivastav', 'Yuvraj Keswani', 'Abishek Porel', 'Yudhajit Guha', 'Pradipta Pramanik', 'Mohammed Shami', 'Sayan Ghosh', 'Akash Deep', 'Kanishk Seth'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Haryana': {'name': ['Mayank Shandilya', 'Arsh Ranga', 'Yuvraj Yogender Singh', 'Ankit Kumar', 'Nishant Sindhu', 'Anshul Kamboj', 'Sumit Kumar', 'Bhuwan Rohilla', 'Anuj Thakral', 'Yuzvendra Chahal', 'Yashvardhan Dalal', 'Parth Vats'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowler', 'Bowler', 'Bowler', '', '']}, 'Puducherry': {'name': ['Pugazhendi Akash', 'Anand Bais', 'Parameeswaran Sivaraman', 'Ragavan Ramamoorthy', 'Sidak Singh', 'Vedant Bhardwaj', 'Aditya Garhwal', 'Thamaraikannan Parandaman', 'Aman Khan', 'Krishna Pandey', 'Ajay Rohera', 'Bhanu Anand', 'Raghav Goyal', 'Himanshu Sharma', 'Jayant Yadav', 'Adil Ayub Tunda', 'A Aravinddaraj', 'Gaurav Yadav', 'Vijai Raja'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Saurashtra': {'name': ['Vishvaraj Jadeja', 'Jay Gohil', 'Siddhant Rana', 'Chirag Jani', 'Prerak Mankad', 'Parswaraj Rana', 'Luckyraj Vaghela', 'Parth Bhut ', 'Sammar Gajjar', 'Harvik Desai', 'Ruchit Ahir', 'Yuvraj Chudasama', 'Dharmendrasinh Jadeja', 'Chetan Sakariya', 'Jaydev Unadkat', 'Krains Fuletra', 'Devang Karamta ', 'Ankur Panwar'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Tripura': {'name': ['Saruk Hossain', 'Joydeep Banik', 'Sankar Paul', 'Sridam Paul', 'Amit Ali', 'Bikramkumar Das', 'Indrajit Debnath', 'Hanuma Vihari', 'Vijay Shankar', 'Tejasvi Jaiswal', 'Arjun Debnath', 'Saurabh Das', 'Manisankar Murasingh', 'Apuraba Biswas', 'Sentu Sarkar', 'Babul Dey', 'Viki Saha', 'Chiranjit Paul', 'Swapnil Singh', 'Samrat Sutradhar'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', '']}, 'Rajasthan': {'name': ['Bharat Sharma', 'Ramnivas Golada', 'Ajayraj Singh', 'Sachin Yadav', 'Shubham Garhwal', 'Deepak Hooda', 'Mahipal Lomror', 'Ram Mohan Chouhan', 'Manav Suthar', 'Sahil Dhiwan', 'Kartik Sharma', 'Kunal Singh Rathore', 'Kukna Ajay Singh', 'Akash Maharaj Singh', 'Ashok Sharma', 'Rahul Chahar', 'Kamlesh Nagarkoti'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Tamil Nadu': {'name': ['Andre Siddarth C', 'Shahrukh Khan', 'Sai Sudharsan', 'Shivam Singh', 'Ravisrinivasan Sai Kishore', 'Sonu Yadav', 'Narayan Jagadeesan', 'Tushar Raheja', 'Amit Sathvik', 'Pradosh Ranjan Paul', 'Rithik Easwaran', 'Varun Chakaravarthy', 'Manimaran Siddharth', 'T Natarajan', 'Gurjapneet Singh', 'R Silambarasan', 'Esakkimuthu A'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'WK-Batsman', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Delhi': {'name': ['Arpit Rana', 'Vaibhav Kandpal', 'Ayush Doseja', 'Sarthak Ranjan', 'Himmat Singh', 'Yash Dhull', 'Priyansh Arya', 'Nitish Rana', 'Mayank Rawat', 'Harsh Tyagi', 'Ayush Badoni', 'Dhruv Kaushik', 'Rahul Dagar', 'Yash Bhatia', 'Rohan Rana', 'Tejasvi Dahiya', 'Anuj Rawat', 'Money Grewal', 'Aryan Rana', 'Suyash Sharma', 'Simarjeet Singh', 'Prince Yadav', 'Ankit Kumar'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler', '']}, 'Jharkhand': {'name': ['Virat Singh', 'Bal Krishna', 'Rajandeep Singh', 'Anukul Roy', 'Pankaj Kumar', 'Utkarsh Singh', 'Ishan Kishan', 'Kumar Kushagra', 'Robin Minz', 'Sushant Mishra', 'Vikash Singh', 'Saurabh Shekhar'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', '']}, 'Karnataka': {'name': ['Mayank Agarwal', 'Karun Nair', 'Smaran Ravichandran', 'Abhinav Manohar', 'Devdutt Padikkal', 'Macneil Noronha', 'Shubhang Hegde', 'Praveen Dubey', 'Krishnan Shrijith', 'Sharath BR', 'Shreyas Gopal', 'Shikhar Shetty', 'Vijaykumar Vyshak', 'Vidhwath Kaverappa', 'Vidyadhar Patil', 'Shreevathsa Acharya'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Bowling Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}, 'Uttarakhand': {'name': ['Bhupen Lalwani', 'Dikshanshu Negi', 'Agrim Tiwari', 'Aarav Mahajan', 'Avneesh Sudha', 'Himanshu Bisht', 'Rahul Raj ', 'Aanjaneya Suryavanshi', 'Yuvraj Chaudhary', 'Mayank Mishra', 'Kunal Chandela', 'Shashwat Dangwal', 'Jagmohan Nagarkoti', 'Prashant Chopra', 'Sanskar Rawat', 'Agnivesh Ayachi', 'Akash Madhwal', 'Rajan Kumar', 'Jagadeesha Suchith'], 'role': ['Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batsman', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Batting Allrounder', 'Bowling Allrounder', 'WK-Batsman', 'WK-Batsman', 'Bowler', 'Bowler', 'Bowler', 'Bowler']}}

print(bcci_squads)
print(cricbuzz_squads)

{'Railways': {'name': ['Shivam Chaudhary', 'S A Ahuja', 'Mohammad Saif', 'Ravi Singh', 'Ashutosh Sharma', 'Shubham Chaubey', 'R Sharma', 'A R Pandey', 'Karn Sharma', 'R K Choudhury', 'Atal Bihari Rai', 'Navneet Virk', 'Upendra Yadav', 'Kunal Yadav', 'K T Marathe'], 'id': ['JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'KQIiiUcVzC0Oeii1yIJeXiGLJbhJOAJ', 'zJLCQiUGyJJeKi1VXIIJeOcib0AiOih', 'JJOzXACeeI1ib0cyiQiLUJGIJiiKOVh', 'JbeIILJUJQGJiA0zeKicyOXihCViO1i', 'ciOIiyOLJiVAeiUbIzeQKX10GJCiJhJ', '20227a4bd4e78d7d11ec9eb102726df', 'A1yhCI0IObKzJicOeJJeLiiQXUVJiiG', 'GiQi1LUCieIAJJJeiJbcOyOXzVi0hKI', 'UIJcCiehyOiziiXKibJQJGIOL1VeJA0', 'yIIJ0KziiVJJhQAeCeiiUiJcbOXO1GL', 'VJ1QiiGe0ziOKCIhJIJOLyiXeicUAbJ', 'iIXOiiJCK0bQJO1zILcVeyAehJiJiGU', 'yAOJiLzUIbXKieJ1IhiVciOJGQCe0Ji', 'OhKzii1cOiyX0CIJVLJAIeiibUJGQeJ']}, 'Mumbai': {'name': ['Shardul Thakur', 'Tushar U Deshpande', 'Shivam Dube', 'Sairaj B Patil', 'Shams Mulani', 'Suryansh Shedge', 'Atharva Vinod Ankolekar', 'Ayush Mhatre', 'Ajinkya Rahane', 'Surya', 'H

In [28]:
from difflib import SequenceMatcher
import re
import difflib

def split_camel_short(name):
    parts, word = [], ""
    for i, ch in enumerate(name):
        word += ch
        if i == len(name)-1 or (i+1 < len(name) and name[i+1].isupper()):
            parts.append(word)
            word = ""
    return parts


def find_full_name(team, short_name):
    try:
        if " (IP" in short_name:
            short_name = short_name.split(' (IP')[0]
        if " (RP" in short_name:
            short_name = short_name.split(' (RP')[0]
        if "(C)" in short_name:
            short_name = short_name.split('(c)')[0]
        if "(Wk)" in short_name:
            short_name = short_name.split('(wk)')[0]  
        if "(C Sub)" in short_name:
            short_name = short_name.split("(C Sub)")[0]  
        if short_name == "R Sharma":
            return "Rahul Sharma"
        if short_name == "Upendra Yadav":
            return short_name
        if short_name == "S N Khan":
            return "Sarfaraz Khan"
        if short_name == "S D Lad":
            return "Siddhesh Lad"
        if short_name == "P M Yadav":
            return "Prateek Yadav"
        if "Sanjeet Desai" in short_name:
            return "Sanjeet Desai"
        if short_name == "A G Rao":
            return 'Anand Rao'
        if short_name == "A G Daga":
            return "Adhyayan Daga"
        if short_name == "A V Wadkar":
            return "Akshay Wadkar"
        if short_name == "R D Gaikwad":
            return "Ruturaj Gaikwad"
        if short_name == "R R Nikam":
            return "Ranjeet Nikam"
        if short_name == "N S Naik":
            return "Nikhil Naik"
        if short_name == "R S Ghosh":
            return "Ramakrishna Ghosh"
        if short_name == "Y J Nahar":
            return "Yash Nahar"
        if short_name == "Koushik V":
            return "Vasuki Koushik"
        if short_name == "E C Sen":
            return "Ekant Sen"
        if short_name == "N R Gangta":
            return "Nikhil Gangta"
        if short_name == "M J Dagar":
            return "Mayank Dagar"
        if short_name == "A P Vashisht":
            return "Akash Vasisht"
        if short_name == "R I Thakur":
            return "Ravi Thakur"
        if short_name == "Amanpreet  V Singh":
            return "Amanpreet Singh"
        if short_name == "N K Kanwar":
            return "Naveen Kanwar"
        if short_name == "Harnoor Singh":
            return "Harnoor Pannu"
        if short_name == "Jass Inder":
            return "Jassinder Singh"
        if short_name == "Abhishek":
            return "Abhishek Tiwari"
        if short_name == "V U Yadav":
            return "Vikas Umesh Yadav"
        if short_name == "Avinash":
            return "Avinash Ramveer"
        if short_name == "H V Patel":
            return "Harshal Patel"
        if short_name == "J K Bhatt":
            return "Japagnya Bhatt"
        if short_name == "Rishi Patel":
            return "Rishi Patel"
        if short_name == "G H Vihari":
            return "Hanuma Vihari"
        if short_name == "M B Mura Singh":
            return "Manisankar Murasingh"
        if short_name == "Tejasvi Jaiswal":
            return short_name
        if short_name == "C K Paul":
            return "Chiranjit Paul"
        if short_name == "D A Jadeja":
            return "Dharmendrasinh Jadeja"
        if short_name == "Gajjar  Sammar":
            return "Sammar Gajjar"
        if short_name == "P Rana":
            return "Parswaraj Rana"
        if short_name == "Madhwal A":
            return "Akash Madhwal"
        if short_name == "Suchith J":
            return "Jagadeesha Suchith"
        if short_name == "A C Ayachi":
            return "Agnivesh Ayachi"
        if short_name == "D Negi":
            return "Dikshanshu Negi"
        if short_name == "K L Shrijith":
            return "Krishnan Shrijith"
        if short_name == "Macneil H N":
            return "Macneil Noronha"
        if short_name == "Avinash Pyla":
            return "Pyla Avinash"
        if short_name == "P Panduranga Raju":
            return "Penmetsa Panduranga Raju"
        if short_name.lower() == "pvsn raju":
            return "Satyanarayana Raju"
        if short_name.lower() == "sdnv prasad":
            return "SDNV Prasad"
        if short_name == "K Nithish Kumar Reddy":
            return "Nitish Kumar Reddy"
        if short_name == "Rohit Sen":
            return "Rhot Mihir Sen"
        if short_name == "Gaurav Choudhary":
            return "Gourav Madan Choudhury"
        if short_name == "Saly Viswanadh":
            return "Saly Samson"
        if "Nidheesh" in short_name:
            return "MD Nidheesh"
        if short_name=="Asif K M":
            return "KM Asif"
        if short_name == "Sharafuddeen N M":
            return "Muhammed Sharafuddeen"
        if short_name == "S Gani":
            return "Sakibul Gani"
        if short_name == "Bhanu":
            return "Bhanu Kumar"
        if short_name == "Md Izhar":
            return "Mohammed Izhar"
        if short_name == "Pratap":
            return "Raghuvendra Pratap Singh "
        if "Raj" in short_name and "bawa" in short_name.lower():
            return "Raj Angad Bawa"
        if short_name == "C Dhindsa":
            return "Chiragvir Dhindsa"
        if short_name == "Ctl Rakshann":
            return "Chinntla Rakshan Readdi"
        if short_name == "Md Arfaz":
            return "Arfaz Ahmed Mohammad"
        if short_name == "Buddhi Rahul":
            return "Rahul Buddhi"
        if short_name == "K Nitesh Reddy":
            return "Nitish Kumar Reddy"
        if short_name == "Md Siraj":
            return "Mohammed Siraj"
        if short_name == "Mickil Jaiswal":
            return short_name
        if short_name == "K H Pandya":
            return "Krunal Pandya"
        if short_name == "J M Sharma":
            return "Jitesh Sharma"
        if short_name == "N A Rathva":
            return "Ninad Ashvinkumar Rathva"
        if short_name == "H H Pandya":
            return "Hardik Pandya"
        if short_name == "S D Patel":
            return "Safvan Patel"
        if short_name == "Saksham Chaudhary":
            return short_name
        if short_name == "A R Easwaran":
            return "Abhimanyu Easwaran"
        if short_name == "Akash Pugazhanthi":
            return "Pugazhendi Akash"
        if short_name == "J J Yadav":
            return "Jayant Yadav"
        if short_name == "Shrikaran":
            return "Aranganathan Shrikaran"
        if short_name == "P Thamaraikannan":
            return "Thamaraikannan Parandaman"
        if short_name == "Vikneshwaran Marimuthu":
            return "Marimuthu Vikneshwaran"
        if short_name == "Md Shafeeq":
            return "Mohamed Safeequddin"
        if short_name == "Sivamurugan M":
            return "Murugaiyan Sivamurugan"
        if short_name == "A R Rana":
            return "Amit Rana"
        if short_name == "Sunny":
            return "Sunny Sandhu"
        if short_name == "Ragupathy Silambarasan":
            return short_name
        if short_name == "A M Singh":
            return "Akash Maharaj Singh"
        if short_name == "B S Sharma":
            return "Bharat Sharma"
        if short_name == "M K Lomror":
            return "Mahipal Lomror"
        if short_name == "M Choudhary":
            return "Mukul Choudhary"
        if short_name == "A B Kookna":
            return "Kukna Ajay Singh"
        if short_name == "R B Chauhan":
            return "Ram Mohan Chouhan"
        
        s = short_name.strip()

        s = re.sub(r'^\(sub\)?\s*', '', s, flags=re.IGNORECASE)
        s = s.strip("() ").strip()

        if s in team:
            return s

        for player in team:
            if len(player) > len(s):
                if all(p in player for p in split_camel_short(s)):
                    return player
            else:
                if all(p in s for p in split_camel_short(player)):
                    return player

        matches = difflib.get_close_matches(
            s.lower(), [p.lower() for p in team], n=1, cutoff=0.7
        )
        scores = [
            (
                player,
                SequenceMatcher(None, s.lower(), player.lower()).ratio()
            )
            for player in team
        ]

        scores.sort(key=lambda x: x[1], reverse=True)

        for player, score in scores[:5]:
            print(f"{player}: {score:.3f}")
            
        if matches:
            return team[[p.lower() for p in team].index(matches[0])]

        return s
    except Exception as e:
        print(f"Error matching {short_name}",e)
        return short_name


In [33]:
final_squads = {}

for team in bcci_squads:
    final_squads[team] = {'bcci_name':[],'name':[],'id':[]}#,'role':[]}
    print(team)
    for player in bcci_squads[team]['name']:
        if team == "J & K":
            cricbuzz_squad = cricbuzz_squads["Jammu and Kashmir"]['name']
        elif team == "Pondicherry":
            cricbuzz_squad = cricbuzz_squads["Puducherry"]['name']
        else:
            cricbuzz_squad = cricbuzz_squads[team]['name']
        corresponding_player = find_full_name(cricbuzz_squad,player)

        bcci_index = bcci_squads[team]['name'].index(player)
        #cricbuzz_index = cricbuzz_squad.index(corresponding_player)
        final_squads[team]['bcci_name'].append(player)
        final_squads[team]['name'].append(corresponding_player)
        final_squads[team]['id'].append(bcci_squads[team]['id'][index])
        #final_squads[team]['role'] = bcci_squads[team]['role'][index]

        print(player,corresponding_player)

        print()
    print()



Railways
Shivam Chaudhary Shivam Chaudhary

Suraj Ahuja: 0.800
Kush Marathe: 0.476
Rahul Sharma: 0.381
Shubham Chaubey: 0.333
Ashutosh Sharma: 0.333
S A Ahuja Suraj Ahuja

Mohammad Saif Mohammad Saif

Ravi Singh Ravi Singh

Ashutosh Sharma Ashutosh Sharma

Shubham Chaubey Shubham Chaubey

R Sharma Rahul Sharma

Akshat R Pandey: 0.800
Akash Pandey: 0.727
Shubham Chaubey: 0.400
Kunal Yadav: 0.381
Karn Sharma: 0.381
A R Pandey Akshat R Pandey

Karn Sharma Karn Sharma

Raj Choudhary: 0.769
Shivam Chaudhary: 0.552
Shubham Chaubey: 0.357
Suraj Ahuja: 0.333
Karn Sharma: 0.333
R K Choudhury Raj Choudhary

Atal Bihari Rai Atal Bihari Rai

Navneet Virk Navneet Virk

Upendra Yadav Upendra Yadav

Kunal Yadav Kunal Yadav

Kush Marathe: 0.783
Akshat R Pandey: 0.385
Kunal Yadav: 0.364
Karn Sharma: 0.364
Akash Pandey: 0.348
K T Marathe Kush Marathe


Mumbai
Shardul Thakur Shardul Thakur

Tushar U Deshpande Tushar Deshpande

Shivam Dube Shivam Dube

Sairaj B Patil Sairaj Patil

Shams Mulani Shams Mulan

In [37]:
for team in (final_squads):
    print(team,final_squads[team])

Railways {'bcci_name': ['Shivam Chaudhary', 'S A Ahuja', 'Mohammad Saif', 'Ravi Singh', 'Ashutosh Sharma', 'Shubham Chaubey', 'R Sharma', 'A R Pandey', 'Karn Sharma', 'R K Choudhury', 'Atal Bihari Rai', 'Navneet Virk', 'Upendra Yadav', 'Kunal Yadav', 'K T Marathe'], 'name': ['Shivam Chaudhary', 'Suraj Ahuja', 'Mohammad Saif', 'Ravi Singh', 'Ashutosh Sharma', 'Shubham Chaubey', 'Rahul Sharma', 'Akshat R Pandey', 'Karn Sharma', 'Raj Choudhary', 'Atal Bihari Rai', 'Navneet Virk', 'Upendra Yadav', 'Kunal Yadav', 'Kush Marathe'], 'id': ['JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'JCULibKVezGQOiJ01ihIIXyJAeiJOci', 'JCULib